# 1. The data

Everything behind the data report: the filings, the CRSP match, the
release texts, and the three independent records of when each release became
public. Run it top to bottom. The acquisition scripts are written to disk by
the cells that hold them, so they are there to read and to run, but they stay
off unless `RUN_FROM_SCRATCH` is set: they need a WRDS account and about a
day of fetching.

What the last round of feedback changed, and where each change is:

**The EDGAR acceptance time no longer comes from the submissions JSON.**
Section 6 shows why, on 300 filings checked against the filing headers. The
JSON disagrees with the header in two cases out of three, and not by a single
offset: in 128 of 300 it is Eastern time wearing a "Z", and in 73 the offset
was added twice to a value that already carried it.
`fetch_acceptance.py` reads `ACCEPTANCE-DATETIME` out of each filing's own
header instead, and `build_event_table.py` prefers it.

**An after-close release lands on the next trading session, not the next
calendar day.** A Friday evening release used to land on a Saturday and then
fall out of the sample when the trading-day filter ran, which cost 5,869
events. It is now rolled forward to the next session, which is what the event
implies, and 7,674 events, 2.3%, move that way at no cost.

**The wire decides, EDGAR is the fallback.** No minimum of the two is taken
anywhere. Taking the earliest of several noisy dates is the one thing the
feedback rules out.

**An event is kept only where two sources put it in the same session.**
`three_source_timing.py` writes the corroborated set by default and reports
what it dropped; `--keep-all` gives the full set for inspection.

**Analyst coverage counts people, not revisions.** It used to sum the daily
counts over the ninety-day window, so one analyst revising twice passed a
filter meant to require three people.

**Every timestamp written to disk carries its offset.** A bare "16:05:00" is
Eastern to the writer and UTC to the next reader, and reading it as UTC moves
a release from after the close into the middle of the session: a four-hour
look-ahead on exactly the events this project is about.

In [1]:
# SETUP
# pandas is the only dependency. Pinning an exact version fights the numpy
# Colab ships with and forces a restart, so this installs only if the
# runtime is older than what the code needs.
%pip install -q "pandas>=2.0"

import glob, os, zipfile
import pandas as pd

# The acquisition pass is off. It needs a WRDS account and about a day of
# fetching from EDGAR, and it is what produced the files in nov2_data.
RUN_FROM_SCRATCH = False
WRDS_USER = ""

print("pandas", pd.__version__)


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\jeanm\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


pandas 3.0.5


In [2]:
#where the files are
#
#two folders and one rule.
#  1_2nov/data  the base files, what we pulled from WRDS and the SEC.
#               nothing is ever written here. (formerly nov2_data)
#  files csv    everything the scripts build, including the rebuilt
#               event_table.csv with the EDGAR times read from the filing
#               headers instead of the submissions JSON.
#
#each part has the same layout as parts 3-5: code/, data/, cache/,
#results/ and logs/. paths come from scripts/project_paths.py, so the
#notebook runs from any folder of the checkout.
#
#the rule is in 1_2nov/code/paths.py: a script reads the version in files csv when there
#is one, and the base version otherwise. that is what stops the loop we had,
#where build_event_table.py rewrote the file fetch_acceptance.py had just
#read and nothing downstream could tell which version it got.

import importlib
import os
import sys
from pathlib import Path

_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts" / "project_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside S08, or set S08_ROOT to the checkout path.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import project_paths
importlib.reload(project_paths)
from project_paths import (REPO_ROOT, TEAM_DIR, NOV2_DATA,
                           PART1_DIR, PART1_CODE, PART2_DIR, PART2_CODE)

# Notebook paths.
CODE = str(PART1_CODE)                     # the pipeline scripts, written below
DATA = str(PART1_DIR / "data")             # base files (formerly nov2_data)
BUILT = str(NOV2_DATA)                     # files csv
RESULTS = str(PART1_DIR / "results")
CACHE = str(PART1_DIR / "cache")           # downloads made by the scripts
LOGS = str(PART1_DIR / "logs")
MANUAL = str(PART2_DIR / "Manual validation")

for d in (CODE, BUILT, RESULTS, CACHE, LOGS, MANUAL):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, CODE)

#the scripts read these two names out of the environment
os.environ["NOV2_BASE"] = DATA
os.environ["NOV2_BUILT"] = BUILT

NEED = ["attrition_funnel.csv", "raw_counts.csv", "matched_sample.csv",
        "extraction_report.csv", "timing_validation.csv", "wire_times.csv",
        "three_source_timing.csv", "event_table.csv"]

if not os.path.isdir(DATA):
    raise FileNotFoundError(
        f"no base data at {DATA}. copy the former nov2_data files "
        f"into 1_2nov/data.")

missing = [f for f in NEED
           if not os.path.isfile(os.path.join(DATA, f))
           or os.path.getsize(os.path.join(DATA, f)) == 0]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} of the {len(NEED)} base files are not in {DATA}: "
        + ", ".join(missing) + ". "
        "Drive sends the small files first, so this is usually an upload "
        "still running.")

#which version of each file a script will actually read
print("base  ", DATA)
print("built ", BUILT)
print()
for f in sorted(set(NEED) | set(os.listdir(BUILT))):
    b = os.path.join(BUILT, f)
    if os.path.isfile(b) and os.path.getsize(b) > 0:
        print(f"{os.path.getsize(b):>12,}  {f:<30} built")
    elif os.path.isfile(os.path.join(DATA, f)):
        print(f"{os.path.getsize(os.path.join(DATA, f)):>12,}  {f:<30} base")

print()
print("results ->", RESULTS)
print("manual  ->", MANUAL)
print("code    ->", CODE)

base   C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\1_2nov\nov2_data
built  C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\files csv

  13,867,718  acceptance_times.csv           built
         483  attrition_funnel.csv           built
 105,538,999  event_table.csv                built
   1,836,959  extraction_report.csv          built
   8,721,316  extraction_report_full.csv     built
  12,350,500  matched_sample.csv             built
         550  raw_counts.csv                 base
  12,921,864  three_source_timing.csv        built
  13,914,223  three_source_timing_all.csv    built
       4,887  timing_conflicts_50.csv        built
      11,445  timing_handcheck_50.csv        built
  10,424,706  timing_validation.csv          built
  16,906,801  wire_times.csv                 base

results -> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\results\1_results
manual  -> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in F

## The pipeline

One cell per script. Running a cell writes the file; it does not run it. The
cell after them runs the lot, in order, when `RUN_FROM_SCRATCH` is on.

In [3]:
%%writefile "{CODE}/funnel_counts.py"
"""
15.S08 — Step 1 of the work plan: raw counts of 8-K Item 2.02 filings.

Produces the first column of the attrition funnel the instructors asked for
at the 2 November milestone. This script answers "how many Item 2.02 filings
exist", nothing else. Matching to CRSP and the per-filter attrition come next,
once the WRDS pull is in place.

Source: the SEC bulk submissions archive, which carries the filing metadata
for every EDGAR filer in one download. The per-CIK API would need thousands of
requests for the same information.

    https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip

The archive is read member by member without extracting it, so the only disk
cost is the 1.6 GB zip itself.

Usage:
    python funnel_counts.py --zip /path/to/submissions.zip --out raw_counts.csv

If --zip points at a file that does not exist, the script downloads it first.
"""

from __future__ import annotations

from paths import BASE, src, dst

import argparse
import csv
import io
import json
import os
import sys
import urllib.request
import zipfile
from collections import Counter, defaultdict

BULK_URL = "https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip"

# The SEC requires a descriptive User-Agent carrying a contact address and
# caps requests at 10 per second. One bulk download stays well inside that.
USER_AGENT = os.environ.get(
    "SEC_USER_AGENT", "MIT 15.S08 project marcusjean501@gmail.com"
)

START_YEAR = 2006
END_YEAR = 2025


def download(url: str, dest: str) -> None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length", 0))
        done = 0
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if total:
                pct = 100 * done / total
                print(f"\r  {done/1e9:.2f} / {total/1e9:.2f} GB  ({pct:.1f}%)",
                      end="", file=sys.stderr, flush=True)
    print(file=sys.stderr)


def iter_filing_tables(zf: zipfile.ZipFile):
    """Yield one parallel-array filing table per archive member.

    Two shapes occur. The primary CIK##########.json nests the table under
    filings.recent. The overflow files CIK##########-submissions-00N.json hold
    the table at the top level. Both are handled.
    """
    for info in zf.infolist():
        name = info.filename
        if not name.endswith(".json") or not name.startswith("CIK"):
            continue
        try:
            with zf.open(info) as fh:
                doc = json.load(fh)
        except (json.JSONDecodeError, zipfile.BadZipFile, OSError):
            yield None, name  # counted as unreadable, not silently dropped
            continue

        if isinstance(doc, dict) and "filings" in doc:
            recent = doc.get("filings", {}).get("recent")
            if recent:
                yield recent, name
        elif isinstance(doc, dict) and "form" in doc:
            yield doc, name


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--zip", required=True, help="path to submissions.zip")
    ap.add_argument("--out", default=dst("raw_counts.csv"))
    args = ap.parse_args()

    if not os.path.exists(args.zip):
        print(f"downloading {BULK_URL}", file=sys.stderr)
        download(BULK_URL, args.zip)

    by_year = Counter()            # Item 2.02 filings per year
    all_8k_by_year = Counter()     # every 8-K, for context
    filers_by_year = defaultdict(set)
    unreadable = 0
    members = 0

    with zipfile.ZipFile(args.zip) as zf:
        for table, name in iter_filing_tables(zf):
            members += 1
            if table is None:
                unreadable += 1
                continue

            forms = table.get("form", [])
            items = table.get("items", [])
            dates = table.get("filingDate", [])
            cik = name.split("-")[0][3:].removesuffix(".json").lstrip("0")

            for i, form in enumerate(forms):
                if form != "8-K":
                    continue
                date = dates[i] if i < len(dates) else ""
                if len(date) < 4:
                    continue
                year = int(date[:4])
                if not (START_YEAR <= year <= END_YEAR):
                    continue
                all_8k_by_year[year] += 1
                # items is a comma-separated string, e.g. "2.02,9.01".
                # Match on the exact code so 12.02 can never match 2.02.
                codes = (items[i] if i < len(items) else "").split(",")
                if "2.02" in [c.strip() for c in codes]:
                    by_year[year] += 1
                    filers_by_year[year].add(cik)

            if members % 20000 == 0:
                print(f"\r  {members:,} filers read", end="",
                      file=sys.stderr, flush=True)

    print(file=sys.stderr)

    with open(args.out, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["year", "item_2_02_filings", "all_8k_filings",
                    "distinct_filers_with_2_02"])
        for year in range(START_YEAR, END_YEAR + 1):
            w.writerow([year, by_year[year], all_8k_by_year[year],
                        len(filers_by_year[year])])
        w.writerow(["TOTAL", sum(by_year.values()), sum(all_8k_by_year.values()), ""])

    print(f"filers read      : {members:,}")
    print(f"unreadable members: {unreadable:,}")
    print(f"Item 2.02 total  : {sum(by_year.values()):,}")
    print(f"all 8-K total    : {sum(all_8k_by_year.values()):,}")
    print(f"written          : {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing funnel_counts.py


In [4]:
%%writefile "{CODE}/build_event_table.py"
"""
15.S08 — event-table skeleton for every 8-K Item 2.02 filing, 2006-2025.

Everything here comes out of the SEC bulk submissions archive, so no crawling
is needed at this stage. One row per filing, carrying what we need before the
WRDS join: identifiers, the acceptance timestamp converted to New York time,
and the trading session the release belongs to.

The text fetch is deliberately NOT done here. It should run after the CRSP
match, so that we only download exhibits for filings that survive the filters.

Session rule, matching the instructor feedback:
  - acceptance at or after 16:00 ET  -> the release is public for the next session
  - acceptance before 09:30 ET       -> same session, pre-open
  - otherwise                        -> same session, intraday
  - filingDate later than the acceptance date -> treated as public at 06:00 ET
    on the filingDate, flagged in `timing_flag`

Usage:
    python build_event_table.py --zip <submissions.zip> --out event_table.csv
"""

from __future__ import annotations

from paths import BASE, src, dst

import argparse
import csv
import json
import os
import sys
import zipfile
from datetime import datetime, timedelta, time
from zoneinfo import ZoneInfo

UTC = ZoneInfo("UTC")
ET = ZoneInfo("America/New_York")

START_YEAR = 2006
END_YEAR = 2025

CLOSE = time(16, 0)
OPEN = time(9, 30)


def parse_acceptance(raw: str):
    """Acceptance as the submissions JSON gives it, read as true UTC.

    This is the fallback and it is known to be unreliable. On a sample of
    300 filings the field agrees with the filing header in 99 cases and
    disagrees in 201: in 128 it is Eastern time with a "Z" on it, and in 73
    it is eight hours ahead of Eastern. The three regimes are mixed inside
    the same year, so nothing here can tell them apart. Use
    `load_header_times` wherever the header has been fetched.
    """
    if not raw:
        return None
    try:
        dt = datetime.fromisoformat(raw.replace("Z", "+00:00"))
    except ValueError:
        return None
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=UTC)
    return dt.astimezone(ET)


def load_header_times(path: str) -> dict[str, datetime]:
    """ACCEPTANCE-DATETIME per accession, from fetch_acceptance.py.

    The header field is Eastern time written as YYYYMMDDHHMMSS with no
    zone, and the instructors say it is the one that does not move.
    """
    if not path or not os.path.exists(path):
        return {}
    out: dict[str, datetime] = {}
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            s = (r.get("acceptance_header_et") or "").strip()
            if len(s) != 14 or not s.isdigit():
                continue
            out[r["accession"]] = datetime.strptime(
                s, "%Y%m%d%H%M%S").replace(tzinfo=ET)
    return out


def session_for(acc_et: datetime | None, filing_date: str):
    """Return (release_et, session_date, when, flag).

    `release_et` is the moment we treat the filing as public, which is the
    acceptance time in every ordinary case. `session_date` is a calendar
    date: an after-close release is rolled one calendar day here, and
    wrds_match_and_funnel.py then rolls it forward onto the trading
    calendar, because a Friday evening release lands on a Saturday and
    Saturday is not a session.
    """
    if acc_et is None:
        return None, filing_date, "unknown", "no_acceptance_time"

    acc_date = acc_et.date().isoformat()
    if filing_date and filing_date > acc_date:
        # EDGAR took the filing but disseminated it the following business
        # morning. The report says we treat that as public at 06:00 ET on
        # the filing date, so the timestamp has to say 06:00 and not the
        # acceptance time, which is the previous evening and would date the
        # release before anybody could see it.
        y, m, d = (int(x) for x in filing_date.split("-"))
        return (datetime(y, m, d, 6, 0, tzinfo=ET), filing_date,
                "pre_open", "filed_after_acceptance")

    t = acc_et.timetz().replace(tzinfo=None)
    if t >= CLOSE:
        nxt = (acc_et + timedelta(days=1)).date().isoformat()
        return acc_et, nxt, "after_close", ""
    if t < OPEN:
        return acc_et, acc_date, "pre_open", ""
    return acc_et, acc_date, "intraday", ""


def iter_tables(zf: zipfile.ZipFile):
    for info in zf.infolist():
        name = info.filename
        if not name.endswith(".json") or not name.startswith("CIK"):
            continue
        try:
            with zf.open(info) as fh:
                doc = json.load(fh)
        except Exception:
            continue
        if isinstance(doc, dict) and "filings" in doc:
            t = doc.get("filings", {}).get("recent")
            if t:
                yield t, name, doc.get("name", "")
        elif isinstance(doc, dict) and "form" in doc:
            yield doc, name, ""


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--zip", required=True)
    ap.add_argument("--out", default=dst("event_table.csv"))
    ap.add_argument("--acceptance", default=src("acceptance_times.csv"),
                    help="ACCEPTANCE-DATETIME per accession, from "
                         "fetch_acceptance.py. Used in preference to "
                         "the submissions JSON wherever it has a row.")
    args = ap.parse_args()

    header_et = load_header_times(args.acceptance)
    if header_et:
        print(f"header acceptance times: {len(header_et):,}", flush=True)
    else:
        print("no header file; falling back to the submissions JSON, "
              "which is wrong for about two filings in three",
              file=sys.stderr)

    rows = 0
    counted = {"header": 0, "json": 0}
    flags = {"": 0, "no_acceptance_time": 0, "filed_after_acceptance": 0}
    when = {"after_close": 0, "pre_open": 0, "intraday": 0, "unknown": 0}

    with zipfile.ZipFile(args.zip) as zf, \
         open(args.out, "w", newline="", encoding="utf-8") as out:
        w = csv.writer(out)
        w.writerow([
            "cik", "company", "accession", "filing_date", "report_date",
            "acceptance_utc", "acceptance_et", "acceptance_source",
            "release_et", "session_date", "when",
            "timing_flag", "items", "primary_document", "filing_dir_url",
        ])

        for table, member, company in iter_tables(zf):
            cik_raw = member.split("-")[0][3:].removesuffix(".json")
            cik = cik_raw.lstrip("0")
            forms = table.get("form", [])
            items = table.get("items", [])
            fdates = table.get("filingDate", [])
            rdates = table.get("reportDate", [])
            accs = table.get("accessionNumber", [])
            accept = table.get("acceptanceDateTime", [])
            prim = table.get("primaryDocument", [])

            for i, form in enumerate(forms):
                if form != "8-K":
                    continue
                fd = fdates[i] if i < len(fdates) else ""
                if len(fd) < 4:
                    continue
                year = int(fd[:4])
                if not (START_YEAR <= year <= END_YEAR):
                    continue
                codes = [c.strip() for c in
                         (items[i] if i < len(items) else "").split(",")]
                if "2.02" not in codes:
                    continue

                acc = accs[i] if i < len(accs) else ""
                raw_acc_dt = accept[i] if i < len(accept) else ""
                acc_et = header_et.get(acc)
                if acc_et is not None:
                    counted["header"] += 1
                else:
                    acc_et = parse_acceptance(raw_acc_dt)
                    counted["json"] += 1
                rel_et, sess, wh, flag = session_for(acc_et, fd)

                nodash = acc.replace("-", "")
                url = (f"https://www.sec.gov/Archives/edgar/data/{cik}/"
                       f"{nodash}/")

                w.writerow([
                    cik, company, acc, fd,
                    rdates[i] if i < len(rdates) else "",
                    raw_acc_dt,
                    acc_et.isoformat() if acc_et else "",
                    "header" if acc in header_et else "json",
                    rel_et.isoformat() if rel_et else "",
                    sess, wh, flag,
                    items[i] if i < len(items) else "",
                    prim[i] if i < len(prim) else "",
                    url,
                ])
                rows += 1
                flags[flag] = flags.get(flag, 0) + 1
                when[wh] = when.get(wh, 0) + 1

    print(f"rows written            : {rows:,}")
    print(f"  from the filing header: {counted['header']:,}")
    print(f"  from the JSON         : {counted['json']:,}")
    print(f"  after close           : {when['after_close']:,}")
    print(f"  pre-open              : {when['pre_open']:,}")
    print(f"  intraday              : {when['intraday']:,}")
    print(f"  unknown time          : {when['unknown']:,}")
    print(f"flagged filed>acceptance: {flags['filed_after_acceptance']:,}")
    print(f"flagged no timestamp    : {flags['no_acceptance_time']:,}")
    print(f"written                 : {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing build_event_table.py


In [5]:
%%writefile "{CODE}/fetch_acceptance.py"
"""read ACCEPTANCE-DATETIME from each filing's own header.

the submissions JSON is wrong about the acceptance time for two filings in
three (see check_acceptance_header.py), and in three different ways, so no
rule fixes it. the header is the field the instructors say does not move.

one request per filing, under the SEC limit of ten a second, and it resumes
from whatever is already in the output file.

    python fetch_acceptance.py
"""

from __future__ import annotations

from paths import BASE, src, dst

import argparse
import csv
import os
import re
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

USER_AGENT = "MIT 15.S08 project marcusjean501@gmail.com"
RATE = 9.0
WORKERS = 8
HEADER_RE = re.compile(r"<ACCEPTANCE-DATETIME>\s*(\d{14})")


class RateLimiter:
    def __init__(self, rate: float):
        self.min_gap = 1.0 / rate
        self.lock = threading.Lock()
        self.next_at = 0.0

    def acquire(self) -> None:
        with self.lock:
            now = time.monotonic()
            wait = max(0.0, self.next_at - now)
            self.next_at = max(now, self.next_at) + self.min_gap
        if wait:
            time.sleep(wait)


def fetch_head(url: str, limiter: RateLimiter, retries: int = 3) -> str | None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    for attempt in range(retries):
        limiter.acquire()
        try:
            with urllib.request.urlopen(req, timeout=45) as r:
                # The header is the first thing in the submission file.
                return r.read(4096).decode("utf-8", "replace")
        except urllib.error.HTTPError as e:
            if e.code in (403, 429, 500, 502, 503, 504):
                time.sleep(min(2 ** attempt, 20))
                continue
            return None
        except Exception:
            time.sleep(min(2 ** attempt, 20))
    return None


def already_done(path: str) -> set[str]:
    if not os.path.exists(path):
        return set()
    with open(path, newline="", encoding="utf-8") as f:
        return {r["accession"] for r in csv.DictReader(f)}


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--events", default=src("event_table.csv"))
    ap.add_argument("--out", default=dst("acceptance_times.csv"))
    ap.add_argument("--workers", type=int, default=WORKERS)
    ap.add_argument("--limit", type=int, default=0)
    args = ap.parse_args()

    seen, rows = set(), []
    with open(args.events, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            a = r["accession"]
            if a and a not in seen:
                seen.add(a)
                rows.append((a, r["cik"]))

    done = already_done(args.out)
    fresh = not done
    if done:
        before = len(rows)
        rows = [r for r in rows if r[0] not in done]
        print(f"resuming: {before - len(rows):,} stored, {len(rows):,} to go",
              flush=True)
    if args.limit:
        rows = rows[:args.limit]
    if not rows:
        print("nothing to do")
        return 0

    limiter = RateLimiter(RATE)
    lock = threading.Lock()
    t0 = time.monotonic()
    n = {"done": 0, "missing": 0}

    out = open(args.out, "a", newline="", encoding="utf-8")
    w = csv.writer(out)
    if fresh:
        w.writerow(["accession", "acceptance_header_et"])

    def work(item) -> None:
        acc, cik = item
        url = (f"https://www.sec.gov/Archives/edgar/data/{str(cik).lstrip('0')}/"
               f"{acc.replace('-', '')}/{acc}.txt")
        head = fetch_head(url, limiter)
        m = HEADER_RE.search(head or "")
        stamp = m.group(1) if m else ""
        with lock:
            w.writerow([acc, stamp])
            n["done"] += 1
            if not stamp:
                n["missing"] += 1
            k = n["done"]
            if k % 2000 == 0:
                out.flush()
                el = time.monotonic() - t0
                rate = k / el
                left = (len(rows) - k) / rate / 3600 if rate else 0
                print(f"  {k:,}/{len(rows):,}  {rate:.1f}/s  "
                      f"{left:.1f} h left  {n['missing']:,} without a header",
                      flush=True)

    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        list(ex.map(work, rows))
    out.close()

    print(f"\n{n['done']:,} fetched in {(time.monotonic() - t0) / 3600:.2f} h")
    print(f"{n['missing']:,} with no ACCEPTANCE-DATETIME in the header")
    print(f"-> {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing fetch_acceptance.py


In [6]:
%%writefile "{CODE}/apply_header_times.py"
"""put the header acceptance times into the event table.

we cannot rebuild from scratch: build_event_table.py needs the 1.6 GB SEC
archive and it is not on this machine any more. we do not need it either.
only the time changes, and fetch_acceptance.py has it, so we rewrite the
four columns that depend on it and leave the rest of the row alone.

the session rule is imported from build_event_table.py, not copied, so
there is one rule in the project.

    python apply_header_times.py
"""

from __future__ import annotations

from paths import BASE, src, dst

import argparse
import csv
import importlib.util
import os
import sys
from datetime import datetime
from zoneinfo import ZoneInfo

ET = ZoneInfo("America/New_York")


def load_rule():
    """session_for, from build_event_table.py, without running its main."""
    spec = importlib.util.spec_from_file_location(
        "bet", os.path.join(os.path.dirname(os.path.abspath(__file__)),
                            "build_event_table.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.session_for, mod.parse_acceptance


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--events", default=os.path.join(BASE,
                                                 "event_table.csv"),
                    help="always the base table: this rebuilds it, so "
                         "reading its own output would compound.")
    ap.add_argument("--acceptance", default=src("acceptance_times.csv"))
    ap.add_argument("--out", default=dst("event_table.csv"))
    ap.add_argument("--require-complete", action="store_true",
                    help="refuse to write unless every filing has a header")
    args = ap.parse_args()

    session_for, parse_acceptance = load_rule()

    header: dict[str, datetime] = {}
    with open(args.acceptance, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            s = (r.get("acceptance_header_et") or "").strip()
            if len(s) == 14 and s.isdigit():
                header[r["accession"]] = datetime.strptime(
                    s, "%Y%m%d%H%M%S").replace(tzinfo=ET)
    print(f"{len(header):,} header timestamps", flush=True)

    with open(args.events, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        cols = list(reader.fieldnames or [])
        for extra in ("acceptance_source", "release_et"):
            if extra not in cols:
                cols.insert(cols.index("session_date"), extra)

        n = {"rows": 0, "header": 0, "json": 0, "moved": 0, "resession": 0}
        with open(args.out, "w", newline="", encoding="utf-8") as out:
            w = csv.DictWriter(out, fieldnames=cols)
            w.writeheader()
            for row in reader:
                n["rows"] += 1
                acc = row["accession"]
                was_session = row.get("session_date", "")
                was_acc = row.get("acceptance_et", "")

                acc_et = header.get(acc)
                if acc_et is not None:
                    n["header"] += 1
                    row["acceptance_source"] = "header"
                else:
                    acc_et = parse_acceptance(row.get("acceptance_utc", ""))
                    n["json"] += 1
                    row["acceptance_source"] = "json"

                rel, sess, when, flag = session_for(acc_et,
                                                    row.get("filing_date", ""))
                row["acceptance_et"] = acc_et.isoformat() if acc_et else ""
                row["release_et"] = rel.isoformat() if rel else ""
                row["session_date"] = sess
                row["when"] = when
                row["timing_flag"] = flag

                if row["acceptance_et"] != was_acc:
                    n["moved"] += 1
                if sess != was_session:
                    n["resession"] += 1
                w.writerow(row)

    if args.require_complete and n["json"]:
        print(f"{n['json']:,} filings still have no header. Finish "
              f"fetch_acceptance.py first, or drop --require-complete.",
              file=sys.stderr)
        return 1

    print(f"rows                      {n['rows']:>9,}")
    print(f"  time from the header    {n['header']:>9,}  "
          f"{100 * n['header'] / max(n['rows'], 1):5.1f}%")
    print(f"  still from the JSON     {n['json']:>9,}")
    print(f"timestamp changed         {n['moved']:>9,}  "
          f"{100 * n['moved'] / max(n['rows'], 1):5.1f}%")
    print(f"session changed           {n['resession']:>9,}  "
          f"{100 * n['resession'] / max(n['rows'], 1):5.1f}%")
    print(f"-> {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing apply_header_times.py


In [7]:
%%writefile "{CODE}/wrds_match_and_funnel.py"
"""
15.S08 — finish the attrition funnel: price, size and analyst-coverage filters.

The first two steps of the funnel are already done without WRDS:
  383,153  raw 8-K Item 2.02 filings      (funnel_counts.py, SEC bulk archive)
  328,317  matched to a PERMNO            (ccm_link.csv, validity window applied)

This script applies the three remaining filters. It works year by year and
pushes the aggregations into SQL, because twenty years of CRSP daily is tens
of millions of rows and does not belong in memory.

Authentication is never handled here. Run `wrds.Connection()` once yourself
and accept the .pgpass file it offers to create; after that this runs
unattended.

    python wrds_match_and_funnel.py
"""

from __future__ import annotations

from paths import BASE, src, dst

import argparse
import sys

import pandas as pd

try:
    import wrds
except ImportError:
    sys.exit("pip install wrds")


# --------------------------------------------------------------------------
# Local step: events joined to PERMNO, no WRDS needed
# --------------------------------------------------------------------------

def matched_events(events_csv: str, link_csv: str) -> tuple[pd.DataFrame, int]:
    ev = pd.read_csv(
        events_csv, dtype={"cik": str},
        usecols=["cik", "accession", "filing_date", "session_date", "when",
                 "timing_flag"],
    )
    ev["session_date"] = pd.to_datetime(ev["session_date"])
    ev["cik"] = ev["cik"].str.lstrip("0")
    n_raw = len(ev)

    lk = pd.read_csv(link_csv, dtype=str)
    lk = lk[lk["LINKPRIM"].isin(["P", "C"])].dropna(subset=["cik"])
    lk["cik"] = lk["cik"].str.lstrip("0")
    lk["LINKDT"] = pd.to_datetime(lk["LINKDT"], errors="coerce")
    # 'E' marks a link that is still current.
    lk["LINKENDDT"] = pd.to_datetime(
        lk["LINKENDDT"].replace("E", "2099-12-31"), errors="coerce"
    ).fillna(pd.Timestamp("2099-12-31"))
    lk = lk.dropna(subset=["LINKDT"])

    j = ev.merge(lk[["cik", "gvkey", "LPERMNO", "LINKDT", "LINKENDDT"]],
                 on="cik", how="inner")
    j = j[(j.session_date >= j.LINKDT) & (j.session_date <= j.LINKENDDT)]
    # A handful of accessions resolve to more than one PERMNO; keep the
    # earliest link start, which is the convention in the CCM documentation.
    j = j.sort_values("LINKDT").drop_duplicates("accession", keep="first")
    j = j.rename(columns={"LPERMNO": "permno"})
    j["permno"] = pd.to_numeric(j["permno"], errors="coerce").astype("Int64")
    j = j.dropna(subset=["permno"])
    return j.reset_index(drop=True), n_raw


# --------------------------------------------------------------------------
# WRDS steps, one year at a time
# --------------------------------------------------------------------------

# The CIZ file crsp.dsf_v2 runs to 2025 where the legacy crsp.dsf stops at
# 2024-12-31, so it is the only one that covers the full holdout. The CIZ
# equivalent of the old shrcd 10/11 and exchcd 1/2/3 screen is spelled out in
# CIZ_COMMON below.
CIZ_COMMON = """
      sharetype = 'NS'
  and securitytype = 'EQTY'
  and securitysubtype = 'COM'
  and usincflg = 'Y'
  and issuertype in ('ACOR', 'CORP')
"""


def nyse_breakpoints(db, years: list[int]) -> pd.DataFrame:
    """20th percentile of NYSE market cap, per year, computed in SQL."""
    q = f"""
        select extract(year from dlycaldt)::int as year,
               percentile_cont(0.20) within group (order by dlycap) as nyse_p20
        from crsp.dsf_v2
        where {CIZ_COMMON}
          and primaryexch = 'N'
          and dlycap is not null
          and extract(year from dlycaldt) between {min(years)} and {max(years)}
        group by 1
        order by 1
    """
    return db.raw_sql(q)


def trading_calendar(db, years: list[int]) -> pd.DatetimeIndex:
    """Every date the US equity market traded, over the sample period.

    build_event_table.py rolls an after-close release to the next calendar
    day, which is what EDGAR's own clock does, and a Friday evening release
    therefore lands on a Saturday. Without this calendar those events are not
    late, they are simply absent: the session they carry matches no row in
    CRSP and they fall out of the funnel as if they had failed a filter.
    They are 5,869 events, 1.8% of the matched sample, and they are
    concentrated on Friday evenings and the eves of holidays, so losing them
    is not neutral.

    One query, because the calendar is a property of the market and not of
    the firms we happen to be holding.
    """
    q = f"""
        select distinct dlycaldt as date
        from crsp.dsf_v2
        where dlycaldt between date '{min(years) - 1}-11-01'
                           and date '{max(years)}-12-31'
        order by 1
    """
    return pd.DatetimeIndex(db.raw_sql(q, date_cols=["date"])["date"])


def roll_to_session(dates: pd.Series, cal: pd.DatetimeIndex) -> pd.Series:
    """The first trading session on or after each date.

    Forward, never backward. A release that is public on Friday evening is
    tradable at Monday's open, so Monday is the session that carries its
    return. Rolling back to Friday would date the reaction before the news.
    """
    i = cal.searchsorted(pd.DatetimeIndex(dates), side="left")
    out = pd.Series(pd.NaT, index=dates.index, dtype="datetime64[ns]")
    ok = i < len(cal)
    out[ok] = cal[i[ok]]
    return out


def crsp_for_year(db, year: int, permnos: list[int]) -> pd.DataFrame:
    """Daily price and size for the permnos we need, one year at a time."""
    lst = ",".join(str(p) for p in permnos)
    q = f"""
        select permno, dlycaldt as date,
               abs(dlyprc) as prc, dlyprc as prc_signed, dlycap as mktcap
        from crsp.dsf_v2
        where {CIZ_COMMON}
          and primaryexch in ('N', 'A', 'Q')
          and permno in ({lst})
          and dlycaldt between date '{year - 1}-11-01' and date '{year}-12-31'
    """
    return db.raw_sql(q, date_cols=["date"])


def ibes_crsp_bridge(db) -> pd.DataFrame:
    """WRDS's own IBES ticker to PERMNO link, with its validity window.

    This replaces joining on an eight-character CUSIP substring, which cannot
    use an index and is heavy enough that the server drops the connection.
    Score 1 to 3 are the matches WRDS considers reliable.
    """
    q = """
        select ticker, permno, sdate, edate
        from wrdsapps.ibcrsphist
        where score <= 3
    """
    return db.raw_sql(q, date_cols=["sdate", "edate"])


def analyst_estimates(db, year: int, tickers: list[str]) -> pd.DataFrame:
    """One row per analyst and announcement date, from the UNADJUSTED detail
    file. The summary file rounds to the penny and restates for splits, which
    Payne and Thomas (2003) show can overturn results.

    This returns the analyst identifiers rather than a count. The filter is
    "at least three analysts covered the firm in the ninety days before the
    release", and an analyst who revised three times in that window is one
    analyst. Counting per date and summing over the window, which is what
    this did before, counted the revisions: a single analyst who revised
    twice passed a filter meant to require three people.
    """
    lst = ",".join("'" + t.replace("'", "''") + "'" for t in tickers)
    q = f"""
        select distinct d.ticker, d.anndats, d.analys
        from ibes.detu_epsus as d
        where d.measure = 'EPS'
          and d.fpi = '6'
          and d.analys is not null
          and d.ticker in ({lst})
          and d.anndats between date '{year - 1}-09-01' and date '{year}-12-31'
    """
    return db.raw_sql(q, date_cols=["anndats"])


# --------------------------------------------------------------------------

class Funnel:
    def __init__(self, n0: int):
        self.rows = [("raw: 8-K Item 2.02 filings 2006-2025", n0, 0, 0.0)]
        self.n = n0

    def step(self, name: str, n_after: int) -> None:
        lost = self.n - n_after
        pct = round(100.0 * lost / self.n, 2) if self.n else 0.0
        self.rows.append((name, n_after, lost, pct))
        self.n = n_after

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows,
                            columns=["step", "remaining", "removed", "pct_removed"])


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--events", default=src("event_table.csv"))
    ap.add_argument("--link", default=src("ccm_link.csv"))
    ap.add_argument("--out-sample", default=dst("matched_sample.csv"))
    ap.add_argument("--out-funnel", default=dst("attrition_funnel.csv"))
    ap.add_argument("--min-analysts", type=int, default=3)
    ap.add_argument("--min-price", type=float, default=5.0)
    ap.add_argument("--user", default=None,
                    help="WRDS username. The password is never passed here; "
                         "it is read from the local pgpass file.")
    args = ap.parse_args()

    ev, n_raw = matched_events(args.events, args.link)
    funnel = Funnel(n_raw)
    funnel.step("matched to a PERMNO, link window respected", len(ev))
    print(f"{n_raw:,} raw  ->  {len(ev):,} matched", flush=True)

    db = wrds.Connection(wrds_username=args.user) if args.user else wrds.Connection()

    years = sorted(ev["session_date"].dt.year.unique())
    years = [y for y in years if 2006 <= y <= 2025]

    cal = trading_calendar(db, years)
    rolled = roll_to_session(ev["session_date"], cal)
    moved = int((rolled.notna() & (rolled != ev["session_date"])).sum())
    off_calendar = int(rolled.isna().sum())
    ev["session_date_raw"] = ev["session_date"]
    ev["session_date"] = rolled
    ev = ev.dropna(subset=["session_date"])
    print(f"rolled to the next session: {moved:,} events "
          f"({100 * moved / max(len(rolled), 1):.2f}%), "
          f"{off_calendar:,} past the end of the calendar", flush=True)
    funnel.step("release session rolled onto the trading calendar", len(ev))

    years = sorted(ev["session_date"].dt.year.unique())
    years = [y for y in years if 2006 <= y <= 2025]

    bp = nyse_breakpoints(db, years)
    print(f"NYSE breakpoints for {len(bp)} years", flush=True)

    bridge = ibes_crsp_bridge(db)
    bridge["edate"] = bridge["edate"].fillna(pd.Timestamp("2099-12-31"))
    print(f"IBES-CRSP bridge: {len(bridge):,} rows", flush=True)

    kept = []
    # Exact per-stage accounting, summed across years, so that the bottom of
    # the funnel equals the number of rows actually written.
    cnt = {"in": 0, "cal": 0, "px": 0, "price": 0, "cap": 0, "cov": 0}

    for year in years:
        sub = ev[ev["session_date"].dt.year == year].copy()
        permnos = sorted(sub["permno"].dropna().unique().tolist())
        if not permnos:
            continue

        cnt["in"] += len(sub)
        px = crsp_for_year(db, year, permnos)
        if px.empty:
            continue

        # t-2 on the market's trading calendar, not on calendar days and not
        # on the subset of dates this year's permnos happened to trade. A
        # calendar built from the holdings themselves shifts whenever the
        # holdings change, which would make the reference date depend on the
        # sample rather than on the market.
        sub["i"] = cal.searchsorted(pd.DatetimeIndex(sub["session_date"]),
                                    side="left")
        sub = sub[(sub["i"] >= 2) & (sub["i"] < len(cal))]
        cnt["cal"] += len(sub)
        sub["ref_date"] = cal[sub["i"].to_numpy() - 2]

        sub = sub.merge(px, left_on=["permno", "ref_date"],
                        right_on=["permno", "date"], how="inner")
        cnt["px"] += len(sub)

        sub = sub[sub["prc"] > args.min_price]
        cnt["price"] += len(sub)

        cut = bp.loc[bp["year"] == year, "nyse_p20"]
        cut = float(cut.iloc[0]) if len(cut) else 0.0
        sub = sub[sub["mktcap"] > cut]
        cnt["cap"] += len(sub)

        if sub.empty:
            continue

        # permno -> IBES ticker, honouring the bridge's validity window
        need = sub[["accession", "permno", "session_date"]].merge(
            bridge, on="permno", how="inner")
        need = need[(need.session_date >= need.sdate)
                    & (need.session_date <= need.edate)]
        tickers = sorted(need["ticker"].dropna().unique().tolist())
        if not tickers:
            continue

        cov = analyst_estimates(db, year, tickers)
        if cov.empty:
            continue
        c = need.merge(cov, on="ticker", how="left")
        c = c[(c.anndats < c.session_date)
              & (c.anndats >= c.session_date - pd.Timedelta(days=90))]
        # Distinct people over the whole window, not the sum of the daily
        # counts, so a revision does not buy a second analyst.
        agg = c.groupby("accession")["analys"].nunique().rename("n_analysts")

        sub = sub.merge(agg, on="accession", how="left")
        sub["n_analysts"] = sub["n_analysts"].fillna(0).astype(int)
        sub = sub[sub["n_analysts"] >= args.min_analysts]
        cnt["cov"] += len(sub)

        kept.append(sub)
        print(f"  {year}: {len(sub):,}", flush=True)

    out = pd.concat(kept, ignore_index=True) if kept else pd.DataFrame()

    funnel.step("inside the t-2 window of the trading calendar", cnt["cal"])
    funnel.step("has a CRSP price at t-2", cnt["px"])
    funnel.step(f"price at t-2 above ${args.min_price:g}", cnt["price"])
    funnel.step("market cap above the NYSE 20th percentile", cnt["cap"])
    funnel.step(f"at least {args.min_analysts} analysts in the prior 90 days",
                cnt["cov"])

    keep_cols = ["accession", "cik", "permno", "gvkey", "filing_date",
                 "session_date", "session_date_raw", "when", "timing_flag",
                 "prc", "mktcap", "n_analysts"]
    if not out.empty:
        out[keep_cols].sort_values("session_date").to_csv(args.out_sample,
                                                          index=False)

    f = funnel.frame()
    f.to_csv(args.out_funnel, index=False)
    print()
    print(f.to_string(index=False))
    print()
    print(f"sample -> {args.out_sample}  ({len(out):,} rows)")
    print(f"funnel -> {args.out_funnel}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing wrds_match_and_funnel.py


In [8]:
%%writefile "{CODE}/fetch_releases.py"
"""
15.S08 — download the earnings release text for the matched sample.

Replaces the first, naive downloader. That one pulled the complete submission
file for every filing, which averages 1.6 MB because it carries the XBRL
exhibits and every embedded image. For 117,926 filings that is roughly 190 GB
of transfer to keep about 19 GB of text, and it was also sequential, so the
throughput ceiling was one filing per second rather than the ten per second
the SEC allows.

This version does two things differently.

  1. It reads the filing index page first (about 10 KB), which lists each
     document with its <TYPE>, then downloads only the exhibit it actually
     wants. Two requests per filing instead of one, but roughly a seventh of
     the bytes.
  2. It runs a small pool of workers behind one shared rate limiter, so the
     ten-per-second ceiling is the binding constraint rather than round-trip
     latency.

Output is one zip per year and the run resumes from whatever is already
stored, so an interruption costs nothing.

    python fetch_releases.py --events matched_sample.csv --out-zip-dir releases
"""

from __future__ import annotations

from paths import BASE, src, dst

import argparse
import csv
import html
import os
import queue
import re
import sys
import threading
import time
import urllib.error
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor

USER_AGENT = "MIT 15.S08 project marcusjean501@gmail.com"
RATE = 9.0          # below the SEC's ceiling of 10 requests a second
WORKERS = 8
MIN_WORDS = 500

ROW_RE = re.compile(r"<tr[^>]*>(.*?)</tr>", re.S | re.I)
CELL_RE = re.compile(r"<td[^>]*>(.*?)</td>", re.S | re.I)
HREF_RE = re.compile(r'href="([^"]+)"', re.I)
SCRIPT_RE = re.compile(r"<(script|style)\b.*?</\1>", re.S | re.I)
TABLE_RE = re.compile(r"<table\b.*?</table>", re.S | re.I)
TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"[ \t\r\f\v]+")
NL_RE = re.compile(r"\n{3,}")


class RateLimiter:
    """One shared gate, so N workers together stay under `rate` per second."""

    def __init__(self, rate: float):
        self.min_gap = 1.0 / rate
        self.lock = threading.Lock()
        self.next_at = 0.0

    def acquire(self) -> None:
        with self.lock:
            now = time.monotonic()
            wait = max(0.0, self.next_at - now)
            self.next_at = max(now, self.next_at) + self.min_gap
        if wait:
            time.sleep(wait)


def fetch(url: str, limiter: RateLimiter, retries: int = 4) -> bytes | None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    for attempt in range(retries):
        limiter.acquire()
        try:
            with urllib.request.urlopen(req, timeout=45) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code in (403, 429, 500, 502, 503, 504):
                time.sleep(min(2 ** attempt, 20))
                continue
            return None
        except Exception:
            time.sleep(min(2 ** attempt, 20))
    return None


def _strip(fragment: str) -> str:
    return html.unescape(TAG_RE.sub(" ", fragment)).replace("\xa0", " ")


def to_text(raw: str) -> tuple[str, int]:
    """Drop financial-statement tables, keep layout tables.

    Older EDGAR filings routinely wrap the whole press release in a <table>
    used purely for layout, so removing every table removes the document. We
    therefore judge each table on its content: one that is mostly digits is a
    statement table and goes, because a statement flattened into a word stream
    is noise for the encoder and the numbers come from I/B/E/S anyway. One
    that reads as prose stays.
    """
    body = SCRIPT_RE.sub(" ", raw)

    dropped = 0

    def decide(m: re.Match) -> str:
        nonlocal dropped
        inner = _strip(m.group(0))
        letters = sum(c.isalpha() for c in inner)
        digits = sum(c.isdigit() for c in inner)
        if digits > 0.30 * max(letters + digits, 1):
            dropped += 1
            return " "
        return " " + inner + " "

    body = TABLE_RE.sub(decide, body)
    body = _strip(body)
    body = NL_RE.sub("\n\n", WS_RE.sub(" ", body))
    return body.strip(), dropped


def looks_like_prose(text: str) -> bool:
    words = text.split()
    if len(words) < MIN_WORDS or text.count(". ") < 10:
        return False
    return sum(c.isdigit() for c in text) / max(len(text), 1) < 0.25


def rank(doc_type: str) -> int:
    t = doc_type.upper()
    if t == "EX-99.1":
        return 0
    if t == "EX-99.2":
        return 1
    if t.startswith("EX-99"):
        return 2
    if t == "8-K":
        return 3
    return 9


def parse_index(body: str, base: str) -> list[tuple[str, str]]:
    """Return [(type, absolute url)] for the documents worth trying."""
    out = []
    for row in ROW_RE.findall(body):
        cells = CELL_RE.findall(row)
        if len(cells) < 4:
            continue
        doc_type = re.sub(r"<[^>]+>", " ", cells[3]).strip()
        if rank(doc_type) == 9:
            continue
        m = HREF_RE.search(cells[2])
        if not m:
            continue
        href = html.unescape(m.group(1))
        # EDGAR serves some documents through an "ix?doc=" viewer wrapper.
        if "ix?doc=" in href:
            href = href.split("ix?doc=", 1)[1]
        url = href if href.startswith("http") else \
            "https://www.sec.gov" + (href if href.startswith("/") else "/" + href)
        out.append((doc_type, url))
    out.sort(key=lambda p: rank(p[0]))
    return out


def one_filing(row: dict, limiter: RateLimiter) -> tuple[dict, str | None]:
    acc = row["accession"]
    cik = str(row["cik"]).lstrip("0")
    base = (f"https://www.sec.gov/Archives/edgar/data/{cik}/"
            f"{acc.replace('-', '')}/")
    rec = {"accession": acc, "cik": cik,
           "filing_date": row.get("filing_date", ""),
           "source": "", "filename": "", "words": 0, "tables_dropped": 0}

    idx = fetch(base + f"{acc}-index.htm", limiter)
    if idx is None:
        rec["source"] = "index fetch failed"
        return rec, None

    for doc_type, url in parse_index(idx.decode("utf-8", "replace"), base):
        body = fetch(url, limiter)
        if body is None:
            continue
        text, n_tables = to_text(body.decode("utf-8", "replace"))
        if looks_like_prose(text):
            rec["source"] = doc_type if doc_type.upper().startswith("EX-99") \
                else "8-K body"
            rec["filename"] = url.rsplit("/", 1)[-1]
            rec["words"] = len(text.split())
            rec["tables_dropped"] = n_tables
            return rec, text

    rec["source"] = "none"
    return rec, None


def already_done(zip_dir: str) -> set[str]:
    done: set[str] = set()
    if not os.path.isdir(zip_dir):
        return done
    for name in sorted(os.listdir(zip_dir)):
        if not name.endswith(".zip"):
            continue
        try:
            with zipfile.ZipFile(os.path.join(zip_dir, name)) as z:
                done.update(n[:-4] for n in z.namelist() if n.endswith(".txt"))
        except zipfile.BadZipFile:
            print(f"  warning: {name} is corrupt, ignoring", file=sys.stderr)
    return done


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--events", default=src("matched_sample.csv"))
    ap.add_argument("--out-zip-dir", default="releases")
    ap.add_argument("--report", default=dst("extraction_report.csv"))
    ap.add_argument("--workers", type=int, default=WORKERS)
    ap.add_argument("--limit", type=int, default=0)
    args = ap.parse_args()

    with open(args.events, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))

    os.makedirs(args.out_zip_dir, exist_ok=True)
    done = already_done(args.out_zip_dir)
    if done:
        before = len(rows)
        rows = [r for r in rows if r["accession"] not in done]
        print(f"resuming: {before - len(rows):,} stored, {len(rows):,} to go",
              flush=True)
    if args.limit:
        rows = rows[:args.limit]

    limiter = RateLimiter(RATE)
    writer_q: queue.Queue = queue.Queue(maxsize=2000)
    report_rows: list[dict] = []
    report_lock = threading.Lock()

    def writer() -> None:
        """Single thread owns the archives; zipfile is not thread safe."""
        zips: dict[str, zipfile.ZipFile] = {}
        while True:
            item = writer_q.get()
            if item is None:
                break
            year, acc, text = item
            if year not in zips:
                zips[year] = zipfile.ZipFile(
                    os.path.join(args.out_zip_dir, f"releases_{year}.zip"),
                    "a", zipfile.ZIP_DEFLATED, compresslevel=6)
            zips[year].writestr(f"{acc}.txt", text)
            writer_q.task_done()
        for z in zips.values():
            z.close()
        writer_q.task_done()

    wt = threading.Thread(target=writer, daemon=True)
    wt.start()

    t0 = time.monotonic()
    counter = {"n": 0}

    def work(row: dict) -> None:
        rec, text = one_filing(row, limiter)
        if text is not None:
            year = (row.get("session_date") or row.get("filing_date") or "")[:4]
            writer_q.put((year or "unknown", rec["accession"], text))
        with report_lock:
            report_rows.append(rec)
            counter["n"] += 1
            n = counter["n"]
        if n % 500 == 0:
            el = time.monotonic() - t0
            rate = n / el
            left = (len(rows) - n) / rate / 3600 if rate else 0
            print(f"  {n:,}/{len(rows):,}  {rate:.1f} filings/s  "
                  f"{left:.1f} h left", flush=True)

    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        list(ex.map(work, rows))

    writer_q.put(None)
    wt.join()

    if not report_rows:
        print("nothing processed")
        return 0

    with open(args.report, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(report_rows[0].keys()))
        w.writeheader()
        w.writerows(report_rows)

    n = len(report_rows)
    counts: dict[str, int] = {}
    for r in report_rows:
        counts[r["source"]] = counts.get(r["source"], 0) + 1
    print(f"\nfilings processed: {n:,} in {(time.monotonic()-t0)/3600:.2f} h")
    for kind, c in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {kind:<22} {c:>7,}  {100*c/n:5.1f}%")
    ok = sorted(r["words"] for r in report_rows if r["words"])
    if ok:
        print(f"word count: median {ok[len(ok)//2]:,}  "
              f"p10 {ok[len(ok)//10]:,}  p90 {ok[(9*len(ok))//10]:,}")
    print(f"report -> {args.report}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing fetch_releases.py


In [9]:
%%writefile "{CODE}/validate_timing.py"
"""
15.S08 - do two independent sources put each release in the same session?

The feedback is blunt about this: no single field gives first dissemination,
EDGAR acceptance can lag the newswire, and taking the earliest of noisy dates
is not allowed. So the EDGAR session is compared against I/B/E/S ANNDATS and
ANNTIMS, which is an independent record of when the firm reported, and the
disagreements are counted rather than resolved by fiat.

Three rules the feedback fixes and this script follows:

  ANNTIMS of 00:00:00 is missing, not midnight.
  ACTDATS and ACTTIMS are never used; they are when I/B/E/S recorded the
  figure, not when the firm said it.
  If EDGAR's filingDate is later than its acceptance date the filing went
  public at 6:00 am ET on the filingDate, which build_event_table already
  applies.

The output is an agreement rate and a drop count, both of which go in the
report, plus a file of the conflicts so fifty can be hand-checked against the
wire.

    python validate_timing.py
"""

from __future__ import annotations

from paths import BASE, src, dst

import pandas as pd
import wrds

def sessions(dates: pd.Series, times: pd.Series) -> pd.DataFrame:
    """The trading session each release at (date, time) belongs to.

    Same rule as the event table: after the close belongs to the next day,
    before the open to the same day. With no time the session is the date, and
    the fact that the time is unknown is carried forward rather than guessed,
    because a release with no time cannot corroborate a session.

    Vectorised, because I/B/E/S returns several hundred thousand actuals and a
    row-wise apply over them costs minutes for no gain. ANNTIMS arrives as a
    string, so it is parsed here rather than trusted.
    """
    d = pd.to_datetime(dates, errors="coerce")
    t = pd.to_datetime(times.astype("string"), format="%H:%M:%S",
                       errors="coerce")
    secs = t.dt.hour * 3600 + t.dt.minute * 60 + t.dt.second

    missing = secs.isna() | (secs == 0)       # 00:00:00 is missing, not midnight
    after = ~missing & (secs >= 16 * 3600)
    pre = ~missing & (secs < 9 * 3600 + 1800)

    sess = d.dt.date.astype("string")
    sess = sess.where(~after, (d + pd.Timedelta(days=1)).dt.date.astype("string"))

    flag = pd.Series("intraday", index=d.index, dtype="string")
    flag = flag.mask(after, "after_close").mask(pre, "pre_open")                .mask(missing, "no_time").mask(d.isna(), "no_date")
    return pd.DataFrame({"ibes_session": sess.where(d.notna()),
                         "ibes_flag": flag})


def main() -> int:
    ev = pd.read_csv(src("matched_sample.csv"),
                     usecols=["accession", "permno", "session_date", "when"],
                     dtype={"accession": str, "permno": "Int64",
                            "session_date": str, "when": str})
    idn = pd.read_csv(src("identity.csv"), usecols=["accession"], dtype=str)
    ev = ev[ev.accession.isin(set(idn.accession))]
    print(f"{len(ev):,} matched events", flush=True)

    db = wrds.Connection(wrds_username="mahaut22")

    print("crsp.dsf_v2 coverage:",
          db.raw_sql("select max(dlycaldt) as d from crsp.dsf_v2")
            .d.iloc[0], flush=True)

    bridge = db.raw_sql("""
        select distinct ticker as ibtic, permno, sdate, edate
        from wrdsapps.ibcrsphist
        where score <= 3
    """, date_cols=["sdate", "edate"])

    # Only the quarterly EPS actual, and only ANNDATS/ANNTIMS. The actuals
    # file carries one row per firm-period, so this is a small table.
    act = db.raw_sql("""
        select ticker as ibtic, pends, anndats, anntims
        from ibes.actu_epsus
        where measure = 'EPS' and pdicity = 'QTR'
          and anndats between '2005-12-01' and '2026-01-31'
    """, date_cols=["pends", "anndats"])
    db.close()
    print(f"ibes actuals {len(act):,}, bridge rows {len(bridge):,}",
          flush=True)

    act["ibtic"] = act.ibtic.str.strip()
    bridge["ibtic"] = bridge.ibtic.str.strip()
    act["permno"] = pd.NA

    a = act.merge(bridge, on="ibtic", how="inner", suffixes=("", "_b"))
    a = a[(a.anndats >= a.sdate) & (a.anndats <= a.edate)]
    a["permno"] = a.permno_b.astype("Int64")

    a = pd.concat([a.reset_index(drop=True),
                   sessions(a.anndats, a.anntims).reset_index(drop=True)],
                  axis=1)

    ev["_d"] = pd.to_datetime(ev.session_date, errors="coerce")
    # An I/B/E/S announcement is matched to the EDGAR event nearest in time,
    # within three calendar days. A wider window would start pairing adjacent
    # quarters and manufacture agreement.
    ev = ev.sort_values("_d")
    a = a.sort_values("anndats")
    m = pd.merge_asof(ev, a[["permno", "anndats", "anntims", "ibes_session",
                             "ibes_flag"]],
                      left_on="_d", right_on="anndats", by="permno",
                      direction="nearest", tolerance=pd.Timedelta("3D"))

    has = m.ibes_session.notna()
    agree = has & (m.ibes_session == m.session_date)
    timed = has & (m.ibes_flag != "no_time")
    agree_timed = timed & (m.ibes_session == m.session_date)

    n = len(m)
    print(f"\nI/B/E/S announcement found within 3 days: {has.sum():,} "
          f"({100*has.mean():.1f}%)")
    print(f"  of those, a usable ANNTIMS:              {timed.sum():,} "
          f"({100*timed.sum()/max(has.sum(),1):.1f}%)")
    print(f"\nsame session, all matched:   {agree.sum():,} "
          f"({100*agree.sum()/max(has.sum(),1):.1f}%)")
    print(f"same session, timed only:    {agree_timed.sum():,} "
          f"({100*agree_timed.sum()/max(timed.sum(),1):.1f}%)")

    # The two-source rule the feedback asks for, costed before it is adopted.
    keep = agree
    print(f"\nif an event is kept only when two sources agree:")
    print(f"  kept    {keep.sum():,} ({100*keep.mean():.1f}% of {n:,})")
    print(f"  dropped {n - keep.sum():,} "
          f"({100*(1-keep.mean()):.1f}%), of which")
    print(f"    no I/B/E/S match   {(~has).sum():,}")
    print(f"    match, different session {(has & ~agree).sum():,}")

    diff = (m.loc[has & ~agree, "_d"]
            - m.loc[has & ~agree, "anndats"]).dt.days
    if len(diff):
        print(f"\nwhen they disagree, EDGAR session minus I/B/E/S date:")
        print(diff.value_counts().head(8).sort_index().to_string())

    m["agrees"] = agree
    cols = ["accession", "permno", "session_date", "when", "anndats",
            "anntims", "ibes_session", "ibes_flag", "agrees"]
    m[cols].to_csv(dst("timing_validation.csv"), index=False)
    con = m.loc[has & ~agree, cols].head(50)
    con.to_csv(dst("timing_conflicts_50.csv"), index=False)
    print("\n-> timing_validation.csv\n-> timing_conflicts_50.csv "
          "(for the hand check against the wire)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing validate_timing.py


In [10]:
%%writefile "{CODE}/wire_times.py"
"""
15.S08 - the wire time, which is what the release time actually is.

The feedback is explicit that release time R is the wire time, that EDGAR is
the fallback when no wire carried the release, and that taking the earliest of
several noisy dates is not acceptable. We had been using EDGAR acceptance for
every event, which is the fallback applied everywhere.

RavenPack's Dow Jones press-release file turns out to be in our entitlement,
one table per year, with a UTC timestamp on every story and an entity
taxonomy. That gives a genuine wire time for the subset of releases Dow Jones
carried, so EDGAR becomes the fallback it was meant to be rather than the only
source.

Three points of care.

  The timestamp is UTC and the session rule is in New York time, so every
  story is converted before anything is compared. This is the same trap the
  feedback flags for the EDGAR submissions JSON.
  Only the earliest story per firm and day is kept. A release generates
  dozens of follow-ups and summaries, and the last of them is not the release.
  The entity is mapped to PERMNO through CUSIP with validity windows, because
  an entity identifier reused after a merger would otherwise attach one firm's
  wire time to another firm's filing.

    python wire_times.py --years 2006-2025
"""

from __future__ import annotations

from paths import BASE, src, dst

import argparse

import pandas as pd
import wrds

# A release can be tagged with any of the earnings categories. Restricting to
# the bare "earnings" tag would drop the releases RavenPack classified by
# direction, which are exactly the informative ones.
EARNINGS_LIKE = "earnings%%"

# Relevance 100 means the story is about this entity rather than mentioning
# it. Anything lower attaches a peer's release time to our filing.
MIN_RELEVANCE = 100


def wire_for_year(db: wrds.Connection, year: int) -> pd.DataFrame:
    """Earliest Dow Jones earnings story per entity and UTC day."""
    sql = f"""
        select rp_entity_id,
               min(timestamp_utc) as wire_utc
        from ravenpack_dj.rpa_djpr_equities_{year}
        where relevance >= {MIN_RELEVANCE}
          and category like '{EARNINGS_LIKE}'
          and news_type = 'PRESS-RELEASE'
        group by rp_entity_id, rpa_date_utc
    """
    try:
        return db.raw_sql(sql)
    except Exception as e:
        # Some vintages lack news_type. Fall back rather than lose the year,
        # and say so, because the looser filter admits wire stories that are
        # not the press release itself.
        print(f"  {year}: retrying without news_type ({str(e).splitlines()[0][:60]})",
              flush=True)
        return db.raw_sql(sql.replace(
            "and news_type = 'PRESS-RELEASE'", ""))


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--years", default="2006-2025")
    ap.add_argument("--out", default=dst("wire_times.csv"))
    args = ap.parse_args()
    lo, hi = (int(x) for x in args.years.split("-"))

    db = wrds.Connection(wrds_username="mahaut22")

    maps = db.raw_sql("""
        select rp_entity_id, cusip, range_start, range_end
        from ravenpack_common.wrds_rpa_all_mappings
        where cusip is not null
    """, date_cols=["range_start", "range_end"])
    maps["cusip8"] = maps.cusip.str.strip().str[:8]
    print(f"entity to cusip rows: {len(maps):,}", flush=True)

    # CRSP historical CUSIP, so a firm that changed CUSIP is matched on the
    # one it carried at the time rather than its last one.
    names = db.raw_sql("""
        select permno, cusip as cusip8, namedt, nameenddt
        from crsp.stocknames_v2
        where cusip is not null
    """, date_cols=["namedt", "nameenddt"])
    names["cusip8"] = names.cusip8.str.strip().str[:8]
    print(f"crsp cusip rows: {len(names):,}", flush=True)

    frames = []
    for y in range(lo, hi + 1):
        w = wire_for_year(db, y)
        w["wire_utc"] = pd.to_datetime(w.wire_utc, errors="coerce", utc=True)
        w = w.dropna(subset=["wire_utc"])
        frames.append(w)
        print(f"  {y}: {len(w):,} entity-days", flush=True)
    db.close()

    wire = pd.concat(frames, ignore_index=True)
    wire["wire_et"] = wire.wire_utc.dt.tz_convert("America/New_York")
    wire["wire_date"] = wire.wire_et.dt.date.astype("string")

    m = wire.merge(maps[["rp_entity_id", "cusip8", "range_start", "range_end"]],
                   on="rp_entity_id", how="inner")
    d = m.wire_et.dt.tz_localize(None)
    m = m[(d >= m.range_start) & (d <= m.range_end.fillna(pd.Timestamp("2100-01-01")))]

    m = m.merge(names, on="cusip8", how="inner")
    d = m.wire_et.dt.tz_localize(None)
    m = m[(d >= m.namedt) & (d <= m.nameenddt)]
    m["permno"] = m.permno.astype("Int64")

    out = (m.groupby(["permno", "wire_date"], as_index=False)
             .wire_et.min()
             .rename(columns={"wire_et": "wire_time_et"}))
    # Written with the offset, not as a bare clock reading. A naive
    # "16:05:00" is Eastern here and UTC to the next reader, and reading it
    # as UTC moves a release from after the close to the middle of the
    # session, which is a look-ahead of four hours on exactly the events
    # the project is about. The offset makes that mistake impossible.
    out["wire_time_et"] = out.wire_time_et.dt.strftime("%Y-%m-%d %H:%M:%S%z")
    out.to_csv(args.out, index=False)

    print(f"\n{len(out):,} firm-days with a Dow Jones wire time")
    print(f"distinct permnos: {out.permno.nunique():,}")
    print(f"-> {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing wire_times.py


In [11]:
%%writefile "{CODE}/three_source_timing.py"
"""
15.S08 - which session does each release belong to, according to all three sources?

EDGAR acceptance, I/B/E/S ANNDATS/ANNTIMS and the Dow Jones wire each imply a
trading session. The feedback asks for an event to be kept only when two
sources agree, and for the drop count to be reported. It also says the release
time is the wire time, so where the wire speaks it decides and EDGAR is the
fallback.

This script does three things and refuses to do a fourth. It computes the
session each source implies, it reports every pattern of agreement rather than
only the headline rate, and it writes the sample of conflicts to hand-check.
It does not pick a winner by taking the earliest timestamp, which is the
specific thing we were told not to do.

    python three_source_timing.py
"""

from __future__ import annotations

from paths import BASE, src, dst

import argparse

import pandas as pd

CLOSE_S = 16 * 3600
OPEN_S = 9 * 3600 + 1800


OFFSET_RE = r"[+-]\d{2}:?\d{2}$|Z$"


def parse_et(s: pd.Series) -> pd.Series:
    """Read a timestamp column and return New York time, naive.

    The column may or may not carry a UTC offset. Older files written by
    wire_times.py do not; current ones do. Passing utc=True unconditionally
    would read a bare "16:05:00" as UTC and move the release to 12:05 New
    York, which is the look-ahead this project exists to avoid. Passing it
    never would fail on a column whose offset changes with daylight saving.
    So the column is inspected before it is parsed.
    """
    txt = s.astype("string").str.strip()
    has_offset = txt.dropna().str.contains(OFFSET_RE, regex=True, na=False)
    if len(has_offset) and has_offset.any():
        t = pd.to_datetime(txt, errors="coerce", utc=True)
        return t.dt.tz_convert("America/New_York").dt.tz_localize(None)
    return pd.to_datetime(txt, errors="coerce")


def to_et(ts: pd.Series) -> pd.Series:
    """New York time, whatever zone the caller handed over.

    The session rule is written in New York hours. Anything tz-aware is
    converted; anything naive is assumed to be New York already, which is
    what the files in this project carry. Without this the hour read off a
    UTC timestamp is four or five hours ahead and a release after the close
    is filed as intraday.
    """
    t = pd.to_datetime(ts, errors="coerce")
    if getattr(t.dtype, "tz", None) is not None:
        t = t.dt.tz_convert("America/New_York").dt.tz_localize(None)
    return t


def session(dates: pd.Series, secs: pd.Series) -> pd.Series:
    """The session implied by a date and a seconds-past-midnight time.

    After the close belongs to the next calendar day, before the open to the
    same one. A missing time yields a missing session rather than a guess,
    because a release with no time cannot corroborate anything.
    """
    d = to_et(dates)
    missing = secs.isna() | (secs == 0)
    after = ~missing & (secs >= CLOSE_S)
    s = d.dt.date.astype("string")
    s = s.where(~after, (d + pd.Timedelta(days=1)).dt.date.astype("string"))
    return s.where(d.notna() & ~missing)


def secs_of(ts: pd.Series) -> pd.Series:
    t = to_et(ts)
    return t.dt.hour * 3600 + t.dt.minute * 60 + t.dt.second


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--drop-json-times", action="store_true",
                    help="drop events whose EDGAR time still comes from "
                         "the submissions JSON. On by default once "
                         "fetch_acceptance.py has covered everything; "
                         "until then it would throw away most of the "
                         "sample, so it is a flag.")
    ap.add_argument("--keep-all", action="store_true",
                    help="write every event, corroborated or not. "
                         "The default keeps only the events two "
                         "sources place in the same session.")
    args = ap.parse_args()

    t = pd.read_csv(src("timing_validation.csv"), dtype={"accession": str})
    w = pd.read_csv(src("wire_times.csv"))

    # we match the wire on the release timestamp, not on the session date.
    # a date rounds everything to midnight, so "nearest" would pick the
    # story closest to midnight instead of the one closest to the release.
    # release_et falls back to acceptance_et on an older event table.
    head = pd.read_csv(src("event_table.csv"), nrows=0).columns
    col = "release_et" if "release_et" in head else "acceptance_et"
    if col != "release_et":
        print("event_table.csv has no release_et column, so the 06:00 rule "
              "for filings disseminated the next morning is not applied "
              "here. Rebuild it with the current build_event_table.py.")
    # acceptance_source says where each EDGAR time came from. The ones from
    # the submissions JSON are wrong for two filings in three and they are
    # carried here so that nothing downstream has to take them on trust.
    want = ["accession", col]
    if "acceptance_source" in head:
        want.append("acceptance_source")
    ev = pd.read_csv(src("event_table.csv"), dtype={"accession": str},
                     usecols=want).rename(columns={col: "release_et"})
    ev = ev.drop_duplicates("accession")
    t = t.merge(ev, on="accession", how="left", validate="many_to_one")
    if "acceptance_source" not in t.columns:
        t["acceptance_source"] = "json"
    print(f"{len(t):,} events, {len(w):,} firm-days with a wire time, "
          f"{t.release_et.notna().sum():,} with a release timestamp")

    from_json = (t.acceptance_source == "json")
    print(f"EDGAR time read from the filing header: "
          f"{(~from_json).sum():,}, still from the JSON: {from_json.sum():,}")
    if args.drop_json_times:
        t = t[~from_json]
        print(f"dropping the JSON ones leaves {len(t):,} events")
    elif from_json.any():
        print("those rows are kept and flagged. Pass --drop-json-times to "
              "exclude them, or finish fetch_acceptance.py first.")

    t["permno"] = pd.to_numeric(t.permno, errors="coerce").astype("Int64")
    w["permno"] = pd.to_numeric(w.permno, errors="coerce").astype("Int64")

    # nearest story on the same firm within three days, same tolerance as
    # I/B/E/S. parse_et reads the offset if the column has one: reading a
    # bare "16:05" as UTC would move the release to 12:05 New York.
    w["wire_dt"] = parse_et(w.wire_time_et)
    t["edgar_dt"] = parse_et(t.release_et)
    t = t.dropna(subset=["edgar_dt"])
    t = t.sort_values("edgar_dt")
    w = w.sort_values("wire_dt").dropna(subset=["wire_dt"])

    m = pd.merge_asof(t, w[["permno", "wire_dt"]], left_on="edgar_dt",
                      right_on="wire_dt", by="permno", direction="nearest",
                      tolerance=pd.Timedelta("3D"))

    m["wire_session"] = session(m.wire_dt, secs_of(m.wire_dt))
    m["ibes_session_c"] = m.ibes_session
    m["edgar_session"] = m.session_date

    has_w = m.wire_session.notna()
    has_i = m.ibes_session_c.notna()
    n = len(m)

    print(f"\ncoverage")
    print(f"  EDGAR session            {n:>8,} (100.0%)")
    print(f"  I/B/E/S session          {has_i.sum():>8,} "
          f"({100*has_i.mean():.1f}%)")
    print(f"  Dow Jones wire session   {has_w.sum():>8,} "
          f"({100*has_w.mean():.1f}%)")
    print(f"  all three                {(has_i & has_w).sum():>8,} "
          f"({100*(has_i & has_w).mean():.1f}%)")

    ei = has_i & (m.edgar_session == m.ibes_session_c)
    ew = has_w & (m.edgar_session == m.wire_session)
    iw = has_i & has_w & (m.ibes_session_c == m.wire_session)

    print(f"\npairwise agreement, among events where both sources exist")
    print(f"  EDGAR with I/B/E/S       {ei.sum():>8,} of {has_i.sum():,} "
          f"({100*ei.sum()/max(has_i.sum(),1):.1f}%)")
    print(f"  EDGAR with the wire      {ew.sum():>8,} of {has_w.sum():,} "
          f"({100*ew.sum()/max(has_w.sum(),1):.1f}%)")
    print(f"  I/B/E/S with the wire    {iw.sum():>8,} of "
          f"{(has_i & has_w).sum():,} "
          f"({100*iw.sum()/max((has_i & has_w).sum(),1):.1f}%)")

    # the wire is the release time where it speaks, EDGAR is the fallback.
    # we settle this first because the rule below is about this column.
    m["session_final"] = m.wire_session.where(has_w, m.edgar_session)
    m["session_source"] = pd.Series("edgar", index=m.index).where(
        ~has_w, "wire")

    # two sources must confirm the session we keep, not just agree with
    # each other. if EDGAR and I/B/E/S say Monday while we keep the wire's
    # Tuesday, that is two sources agreeing and none confirming.
    c_e = m.edgar_session == m.session_final
    c_i = has_i & (m.ibes_session_c == m.session_final)
    c_w = has_w & (m.wire_session == m.session_final)
    confirm = c_e.astype(int) + c_i.astype(int) + c_w.astype(int)
    two = confirm >= 2
    m["n_confirm"] = confirm
    m["agree_two"] = two

    pairs_only = ((ei.astype(int) + ew.astype(int) + iw.astype(int)) >= 1)
    waved = (pairs_only & ~two).sum()

    print(f"\nkeeping an event only when two sources confirm the session we "
          f"keep")
    print(f"  kept     {two.sum():>8,} ({100*two.mean():.1f}%)")
    print(f"  dropped  {n - two.sum():>8,} ({100*(1-two.mean()):.1f}%)")
    print(f"  of which {waved:>8,} would have passed on pairwise agreement, "
          f"with the pair agreeing on a session we do not keep")

    only_edgar = ~has_i & ~has_w
    print(f"    of the dropped, with no second source at all: "
          f"{(only_edgar).sum():,}")
    print(f"    of the dropped, with a second source that disagrees: "
          f"{(n - two.sum() - only_edgar.sum()):,}")

    moved = (m.session_final != m.edgar_session).sum()
    print(f"\nadopting the wire where it exists moves {moved:,} events "
          f"({100*moved/n:.1f}%) to another session")

    print("\nhow far the wire moves the session, in sessions")
    delta = (pd.to_datetime(m.session_final, errors="coerce")
             - pd.to_datetime(m.edgar_session, errors="coerce")).dt.days
    print(delta[delta != 0].value_counts().head(8).sort_index().to_string())

    cols = ["accession", "permno", "edgar_session", "when", "ibes_session_c",
            "ibes_flag", "wire_dt", "wire_session", "session_final",
            "session_source", "n_confirm", "agree_two",
            "acceptance_source"]

    # The corroborated set is what the rest of the project reads. Keeping
    # everything and offering the agreeing subset as an option puts the
    # burden the wrong way round: an event no second source confirms is an
    # event whose session we are guessing, and the guess would reach the
    # return windows unless somebody remembered to filter.
    m[cols].to_csv(dst("three_source_timing_all.csv"), index=False)
    kept = m if args.keep_all else m[two]
    kept[cols].to_csv(dst("three_source_timing.csv"), index=False)
    print(f"\nthree_source_timing.csv holds {len(kept):,} events"
          + ("" if args.keep_all else
             f", {n - len(kept):,} dropped for want of a second source"))

    # Fifty events for the hand check against the wire, drawn from the
    # conflicts rather than at random: a random fifty would be almost all
    # agreements and would tell us nothing we do not already know. Seeded,
    # so the same fifty come back on a rerun.
    conflict = m[has_w & has_i & ~(ei & ew & iw)]
    take = conflict.sample(min(50, len(conflict)), random_state=20261104)
    hand = take[cols].copy()
    hand["edgar_url"] = [
        "https://www.sec.gov/Archives/edgar/data/"
        + f"{a.replace('-', '')}/{a}-index.htm" for a in hand.accession]
    for c in ["session_you_read", "source_you_trust", "note"]:
        hand[c] = ""
    hand.to_csv(dst("timing_handcheck_50.csv"), index=False,
                encoding="utf-8-sig")
    print("-> three_source_timing_all.csv\n-> three_source_timing.csv"
          "\n-> timing_handcheck_50.csv")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing three_source_timing.py


In [12]:
%%writefile "{CODE}/check_access.py"
"""
15.S08 - what our WRDS account can actually reach.

The feedback asks us to confirm four things before November 9 rather than
assume them: the I/B/E/S Guidance entitlement, the SEC linking tables,
RavenPack, and which CRSP tier we have. Each is checked by selecting one row,
because an entitlement that renders in the web interface can still refuse a
query, and a schema that exists can still be empty for us.

    python check_access.py
"""

from __future__ import annotations

import wrds

CHECKS = [
    ("CRSP daily, CIZ", "select max(dlycaldt) as v from crsp.dsf_v2"),
    ("CRSP daily, legacy", "select max(date) as v from crsp.dsf"),
    ("I/B/E/S Guidance", "select count(*) as v from ibes.det_guidance "
                         "where anndats >= '2024-01-01'"),
    ("I/B/E/S Guidance cols", "select * from ibes.det_guidance limit 1"),
    ("I/B/E/S actuals, unadj", "select max(anndats) as v "
                               "from ibes.actu_epsus"),
    ("I/B/E/S detail, unadj", "select max(anndats) as v "
                              "from ibes.detu_epsus"),
    ("IBES-CRSP bridge", "select count(*) as v from wrdsapps.ibcrsphist"),
    ("CCM link", "select count(*) as v from crsp.ccmxpf_lnkhist"),
    ("SEC analytics suite", "select count(*) as v "
                            "from wrdssec.wciklink_gvkey"),
    ("RavenPack equities", "select count(*) as v from ravenpack.rpa_djpr_equities "
                           "limit 1"),
    ("RavenPack, any", "select count(*) as v "
                       "from information_schema.tables "
                       "where table_schema like 'ravenpack%'"),
    ("Loughran-McDonald", "select count(*) as v "
                          "from information_schema.tables "
                          "where table_schema like '%loughran%' "
                          "or table_name like '%loughran%'"),
]


def main() -> int:
    db = wrds.Connection(wrds_username="mahaut22")
    for name, sql in CHECKS:
        try:
            df = db.raw_sql(sql)
            if "v" in df.columns:
                print(f"  OK      {name:<24} {df.v.iloc[0]}")
            else:
                print(f"  OK      {name:<24} {len(df.columns)} columns: "
                      f"{', '.join(list(df.columns)[:12])}")
        except Exception as e:
            msg = str(e).split("\n")[0][:90]
            print(f"  DENIED  {name:<24} {msg}")
    db.close()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing check_access.py


In [13]:
# THE ACQUISITION PASS, IN ORDER, WHEN IT IS TURNED ON ABOVE
import subprocess, sys

PIPELINE = [
    'funnel_counts.py',
    'build_event_table.py',
    'fetch_acceptance.py',
    'apply_header_times.py',
    'wrds_match_and_funnel.py',
    'fetch_releases.py',
    'validate_timing.py',
    'wire_times.py',
    'three_source_timing.py',
    'check_access.py'
]

# arguments each script needs. funnel_counts.py downloads the SEC archive into
# the cache when it is missing, and build_event_table.py reuses it. the release
# texts go to 1_2nov/data/releases_text, so a rerun only fetches what is missing.
ZIP = os.path.join(CACHE, "submissions.zip")
ARGS = {
    'funnel_counts.py': ["--zip", ZIP],
    'build_event_table.py': ["--zip", ZIP],
    'fetch_releases.py': ["--out-zip-dir", os.path.join(DATA, "releases_text")],
}

if RUN_FROM_SCRATCH:
    for script in PIPELINE:
        print("=" * 60, script, flush=True)
        subprocess.run([sys.executable, os.path.join(CODE, script),
                        *ARGS.get(script, [])],
                       check=True, cwd=CACHE)
else:
    for script in PIPELINE:
        print("written, not run:", script)

written, not run: funnel_counts.py
written, not run: build_event_table.py
written, not run: fetch_acceptance.py
written, not run: apply_header_times.py
written, not run: wrds_match_and_funnel.py
written, not run: fetch_releases.py
written, not run: validate_timing.py
written, not run: wire_times.py
written, not run: three_source_timing.py
written, not run: check_access.py


## 5. Every figure in the report, recomputed

Each number is computed again from the event-level files and printed beside
what the report prints. The cell at the end fails if any of them disagree,
so this is a test and not a display.

The report it checks is the revision of 8 November, and the files it reads
are the rebuilt ones in `files csv` where they exist, the base ones in
`nov2_data` otherwise. So a figure that disagrees here is a real break, not
the correction showing through. The version submitted on 2 November sits in
`report/` under its own name and is not checked: thirty-eight of its
fifty-one figures moved, and `1_2nov/results/figures_vs_report.csv` lists
which.

In [14]:
ok, bad = 0, []


def pick(name):
    # the rebuilt file when there is one, the base file otherwise. same rule
    # as paths.py, so this cell checks the report against what the rest of
    # the pipeline actually reads.
    built = os.path.join(BUILT, name)
    return built if os.path.exists(built) else os.path.join(DATA, name)


def check(label, got, want, pct=None):
    global ok
    hit = (round(got, 4) == round(want, 4))
    if hit:
        ok += 1
    else:
        bad.append(f"{label}: got {got}, report says {want}")
    tail = f"  ({pct:.1f}%)" if pct is not None else ""
    print(f"  {'ok ' if hit else 'NO '}{label:<52}{got:>12,}{tail}")

In [15]:
print("1. raw counts")
rc = pd.read_csv(pick("raw_counts.csv"))
# The file carries its own TOTAL row; the yearly rows are the data.
total = rc[rc.year == "TOTAL"].iloc[0]
rc = rc[rc.year != "TOTAL"].astype({"year": int})
check("TOTAL row agrees with the sum of the years",
      int(rc.item_2_02_filings.sum()), int(total.item_2_02_filings))
check("Item 2.02 filings, 2006-2025", int(rc.item_2_02_filings.sum()), 383153)
check("all 8-K filings", int(rc.all_8k_filings.sum()), 1549567)
check("fewest distinct filers in a year",
      int(rc.distinct_filers_with_2_02.min()), 4277)
check("most distinct filers in a year",
      int(rc.distinct_filers_with_2_02.max()), 6003)
check("years covered", int(rc.year.nunique()), 20)
first, last = rc.set_index("year").item_2_02_filings.loc[[2006, 2025]]
check("fall in Item 2.02 count, 2006 to 2025, in percent",
      round(100 * (1 - last / first)), 27)

1. raw counts
  ok TOTAL row agrees with the sum of the years               383,153
  ok Item 2.02 filings, 2006-2025                             383,153
  ok all 8-K filings                                        1,549,567
  ok fewest distinct filers in a year                           4,277
  ok most distinct filers in a year                             6,003
  ok years covered                                                 20
  ok fall in Item 2.02 count, 2006 to 2025, in percent             27


In [16]:
print("\n2. match attrition")
fn = pd.read_csv(pick("attrition_funnel.csv"))
ms = pd.read_csv(pick("matched_sample.csv"))
check("events after the last stage", int(fn.remaining.iloc[-1]), 115561)
check("matched sample rows", len(ms), 115561)
rem = fn.remaining.to_numpy()
check("stages where removed equals the drop in remaining",
      int(sum(int(fn.removed.iloc[i]) == rem[i - 1] - rem[i]
              for i in range(1, len(fn)))), len(fn) - 1)
check("largest single removal, the size cut", int(fn.removed.max()), 81231)
check("median market capitalisation, $bn",
      round(ms.mktcap.median() / 1e6, 1), 3.5)
check("median number of analysts", int(ms.n_analysts.median()), 7)
check("events in 2025, lost with the legacy CRSP file",
      int((pd.to_datetime(ms.session_date).dt.year == 2025).sum()), 5664)


2. match attrition


  ok events after the last stage                              115,561
  ok matched sample rows                                      115,561
  ok stages where removed equals the drop in remaining              7
  ok largest single removal, the size cut                      81,231
  ok median market capitalisation, $bn                            3.5
  ok median number of analysts                                      7
  ok events in 2025, lost with the legacy CRSP file             5,664


In [17]:
print("\n3. release text")
ex = pd.read_csv(pick("extraction_report.csv"))
src = ex.source.fillna("")
got = src.ne("none") & src.ne("index fetch failed")
check("events with a release recovered", int(got.sum()), 114578,
      100 * got.mean())
v1 = src.str.startswith("EX-99.1")
other = src.str.startswith("EX-99") & ~v1
check("EX-99.1 and its variants", int(v1.sum()), 101688, 100 * v1.mean())
check("other EX-99 exhibits", int(other.sum()), 10970, 100 * other.mean())
body = src.eq("8-K body")
check("the 8-K body itself", int(body.sum()), 1920, 100 * body.mean())
check("no prose document found", int(src.eq("none").sum()), 981,
      100 * src.eq("none").mean())
check("index page unreachable", int(src.eq("index fetch failed").sum()), 2)
check("share lost by taking EX-99.1 alone, in percent",
      round(100 * (got.sum() - v1.sum()) / got.sum(), 1), 11.2)
w = ex.loc[got, "words"]
check("median release, words", int(w.median()), 3161)
check("tenth percentile, words", int(w.quantile(0.10)), 1425)
check("ninetieth percentile, words", round(w.quantile(0.90)), 5978)
check("median tables dropped", int(ex.loc[got, "tables_dropped"].median()), 3)


3. release text
  ok events with a release recovered                          114,578  (99.1%)
  ok EX-99.1 and its variants                                 101,688  (88.0%)
  ok other EX-99 exhibits                                      10,970  (9.5%)
  ok the 8-K body itself                                        1,920  (1.7%)
  ok no prose document found                                      981  (0.8%)
  ok index page unreachable                                         2
  ok share lost by taking EX-99.1 alone, in percent              11.2
  ok median release, words                                      3,161
  ok tenth percentile, words                                    1,425
  ok ninetieth percentile, words                                5,978
  ok median tables dropped                                          3


In [18]:
print("\n4. release timing")
CLOSE_S, OPEN_S = 16 * 3600, 9 * 3600 + 1800


def session(dates, secs):
    d = pd.to_datetime(dates, errors="coerce")
    missing = secs.isna() | (secs == 0)
    after = ~missing & (secs >= CLOSE_S)
    s = d.dt.date.astype("string")
    s = s.where(~after, (d + pd.Timedelta(days=1)).dt.date.astype("string"))
    return s.where(d.notna() & ~missing)


def secs_of(ts):
    t = pd.to_datetime(ts, errors="coerce")
    return t.dt.hour * 3600 + t.dt.minute * 60 + t.dt.second


t = pd.read_csv(pick("timing_validation.csv"))
wi = pd.read_csv(pick("wire_times.csv"))
n = len(t)
check("firm-days with a Dow Jones wire time", len(wi), 444915)
check("distinct securities on the wire file", int(wi.permno.nunique()), 8853)

has_i_all = t.ibes_flag.notna()
check("events with an I/B/E/S announcement", int(has_i_all.sum()), 101706,
      100 * has_i_all.mean())
check("EDGAR and I/B/E/S on the same session", int(t.agrees.sum()), 96827,
      100 * t.agrees.sum() / has_i_all.sum())
check("kept if two sources must agree, in percent",
      round(100 * t.agrees.mean(), 1), 85.6)
check("dropped by that rule", int(n - t.agrees.sum()), 16253)
check("  of those, no I/B/E/S match", int((~has_i_all).sum()), 11374)
dis = (~t.agrees) & has_i_all
check("  of those, matched to a different session", int(dis.sum()), 4879)

d = t[dis].copy()
d["lag"] = (pd.to_datetime(d.session_date)
            - pd.to_datetime(d.anndats)).dt.days
same = d.lag == 0
check("same date, I/B/E/S after the close, EDGAR in session",
      int((same & d.when.eq("intraday")).sum()), 199)
check("same date, I/B/E/S after the close, EDGAR pre-open",
      int((same & d.when.eq("pre_open")).sum()), 187)
check("same date, both after the close, different session",
      int((same & d.when.eq("after_close")).sum()), 36)
check("same date, differing only on the side of the close",
      int(same.sum()), 422)
check("EDGAR session one to three days after the I/B/E/S date",
      int(d.lag.between(1, 3).sum()), 4211)
check("EDGAR session before the I/B/E/S date", int((d.lag < 0).sum()), 246)

t["permno"] = pd.to_numeric(t.permno, errors="coerce").astype("Int64")
wi["permno"] = pd.to_numeric(wi.permno, errors="coerce").astype("Int64")
wi["wire_dt"] = pd.to_datetime(wi.wire_time_et, errors="coerce")
t["edgar_dt"] = pd.to_datetime(t.session_date, errors="coerce")
m = pd.merge_asof(t.sort_values("edgar_dt"),
                  wi.sort_values("wire_dt").dropna(subset=["wire_dt"])
                    [["permno", "wire_dt"]],
                  left_on="edgar_dt", right_on="wire_dt", by="permno",
                  direction="nearest", tolerance=pd.Timedelta("3D"))
m["wire_session"] = session(m.wire_dt, secs_of(m.wire_dt))
has_w, has_i = m.wire_session.notna(), m.ibes_session.notna()
check("events given a wire time", int(has_w.sum()), 86538, 100 * has_w.mean())

ei = has_i & (m.session_date == m.ibes_session)
ew = has_w & (m.session_date == m.wire_session)
iw = has_i & has_w & (m.ibes_session == m.wire_session)
check("I/B/E/S agrees with the wire", int(iw.sum()), 82650,
      100 * iw.sum() / (has_i & has_w).sum())
check("  out of events where both exist", int((has_i & has_w).sum()), 83829)
check("EDGAR agrees with the wire", int(ew.sum()), 82363,
      100 * ew.sum() / has_w.sum())
check("EDGAR agrees with I/B/E/S", int(ei.sum()), 96827,
      100 * ei.sum() / has_i.sum())

two = (ei.astype(int) + ew.astype(int) + iw.astype(int)) >= 1
check("retained when two of three sources agree", int(two.sum()), 102726,
      100 * two.mean())
check("dropped by that rule", int(n - two.sum()), 10354)
only_e = ~has_i & ~has_w
check("  of those, with no second source", int(only_e.sum()), 8665)
check("  of those, with a second source that disagrees",
      int(n - two.sum() - only_e.sum()), 1689)

m["session_final"] = m.wire_session.where(has_w, m.session_date)
moved = m.session_final != m.session_date
check("events the wire moves to another session", int(moved.sum()), 4175,
      100 * moved.mean())
delta = (pd.to_datetime(m.session_final, errors="coerce")
         - pd.to_datetime(m.session_date, errors="coerce")).dt.days
check("  EDGAR one to three sessions too early", int((delta > 0).sum()), 574)
check("  EDGAR one to three sessions too late", int((delta < 0).sum()), 3601)


4. release timing


  ok firm-days with a Dow Jones wire time                     444,915
  ok distinct securities on the wire file                       8,853
  ok events with an I/B/E/S announcement                      101,706  (89.9%)
  ok EDGAR and I/B/E/S on the same session                     96,827  (95.2%)
  ok kept if two sources must agree, in percent                  85.6
  ok dropped by that rule                                      16,253
  ok   of those, no I/B/E/S match                              11,374
  ok   of those, matched to a different session                 4,879
  ok same date, I/B/E/S after the close, EDGAR in session         199
  ok same date, I/B/E/S after the close, EDGAR pre-open           187
  ok same date, both after the close, different session            36
  ok same date, differing only on the side of the close           422
  ok EDGAR session one to three days after the I/B/E/S date       4,211
  ok EDGAR session before the I/B/E/S date                        246


  ok events given a wire time                                  86,538  (76.5%)
  ok I/B/E/S agrees with the wire                              82,650  (98.6%)
  ok   out of events where both exist                          83,829
  ok EDGAR agrees with the wire                                82,363  (95.2%)
  ok EDGAR agrees with I/B/E/S                                 96,827  (95.2%)
  ok retained when two of three sources agree                 102,726  (90.8%)
  ok dropped by that rule                                      10,354
  ok   of those, with no second source                          8,665
  ok   of those, with a second source that disagrees            1,689
  ok events the wire moves to another session                   4,175  (3.7%)
  ok   EDGAR one to three sessions too early                      574
  ok   EDGAR one to three sessions too late                     3,601


In [19]:
print(f"{ok} of {ok + len(bad)} figures reproduce the report")
for line in bad:
    print("  MOVED " + line)
assert not bad, "a figure in the report does not reproduce"

pd.DataFrame({"figures": [ok + len(bad)], "reproduce": [ok],
              "moved": [len(bad)]}).to_csv(
    os.path.join(RESULTS, "figure_check.csv"), index=False)

51 of 51 figures reproduce the report


In [20]:
# 6. WHY THE ACCEPTANCE TIME WAS RE-READ
#
# nov2_data holds the 29-30 October run, on the submissions-JSON times, and
# that is what the version of the report submitted on 2 November describes.
# files csv holds the 8 November rerun, on the acceptance times read from the
# filing headers, and that is what the revised report describes and what
# section 5 checks. This section is the measurement that forced the rerun.

chk = pd.read_csv(pick("acceptance_header_check.csv"))
got = chk.dropna(subset=["delta_hours"])
same = (got.delta_hours == 0)

print("EDGAR acceptance time: the submissions JSON against the filing header")
print(f"  filings checked                 {len(got):>6,}")
print(f"  JSON agrees with the header     {same.sum():>6,}  "
      f"{100 * same.mean():5.1f}%")
print(f"  JSON disagrees                  {(~same).sum():>6,}  "
      f"{100 * (~same).mean():5.1f}%")
print("\n  what the field actually holds, decided filing by filing")
for name, n in chk.regime.value_counts().items():
    print(f"    {n:>5,}  {name}")
print("""
    true UTC                 the field is what it claims to be
    Eastern labelled Z       the clock reading is Eastern and the Z is a
                             lie, so converting it moves the release four
                             or five hours too early
    converted to UTC twice   the offset was added on top of a value that
                             already carried it, which lands eight hours
                             past Eastern in summer and ten in winter""")
print("  Three regimes mixed inside the same year, so no rule recovers the")
print("  truth from the JSON alone. fetch_acceptance.py reads the header.")

print("\nThe rerun, done on 8 November, in this order:")
for s in ["fetch_acceptance.py", "apply_header_times.py",
          "wrds_match_and_funnel.py", "validate_timing.py",
          "three_source_timing.py"]:
    print("   ", s)

ev = pd.read_csv(pick("event_table.csv"), usecols=["acceptance_source"])
hdr = (ev.acceptance_source == "header").sum()
print(f"\n  event_table.csv: {hdr:,} of {len(ev):,} rows "
      f"({100 * hdr / len(ev):.1f}%) on a header time")
print("  the rest are filings whose header carries no such field.")
print("  they keep the JSON time and stay flagged, and")
print("  --drop-json-times excludes them.")

print("\nThe release texts did not need refetching for the events")
print("already in: which exhibit a release sits in does not depend on")
print("the timestamp. The corrected sessions did bring 2,481 new events")
print("into the sample, and those were fetched. The CRSP windows were")
print("recomputed in full.")

EDGAR acceptance time: the submissions JSON against the filing header
  filings checked                    300
  JSON agrees with the header         99   33.0%
  JSON disagrees                     201   67.0%

  what the field actually holds, decided filing by filing
      128  Eastern labelled Z
       99  true UTC
       73  converted to UTC twice

    true UTC                 the field is what it claims to be
    Eastern labelled Z       the clock reading is Eastern and the Z is a
                             lie, so converting it moves the release four
                             or five hours too early
    converted to UTC twice   the offset was added on top of a value that
                             already carried it, which lands eight hours
                             past Eastern in summer and ten in winter
  Three regimes mixed inside the same year, so no rule recovers the
  truth from the JSON alone. fetch_acceptance.py reads the header.

The rerun, done on 8 November, i


  event_table.csv: 381,991 of 383,153 rows (99.7%) on a header time
  the rest are filings whose header carries no such field.
  they keep the JSON time and stay flagged, and
  --drop-json-times excludes them.

The release texts did not need refetching for the events
already in: which exhibit a release sits in does not depend on
the timestamp. The corrected sessions did bring 2,481 new events
into the sample, and those were fetched. The CRSP windows were
recomputed in full.


In [21]:
# 7. THE FIFTY TIMING EVENTS, SETTLED
#
# The feedback asks for fifty events checked by hand where the sources
# disagree about the session. The question looks like a judgement call and
# is not one: the filing header carries ACCEPTANCE-DATETIME in Eastern
# time, the instructors say it is the field that does not move, and the
# session follows from the time by the same arithmetic the pipeline uses.
#
# So adjudicate_timing_50.py refetches the header for each of the fifty,
# recomputes the session, and marks every source that matches it. A human
# still owns the verdict column and can overwrite any row; the point is
# that the sheet arrives with an answer and a reason rather than blank.

import subprocess, sys, shutil

sheet = os.path.join(BUILT, "timing_handcheck_50.csv")
if not os.path.exists(sheet):
    print("run three_source_timing.py first: it draws the fifty")
else:
    r = subprocess.run([sys.executable,
                        os.path.join(CODE, "adjudicate_timing_50.py")],
                       capture_output=True, text=True)
    print(r.stdout[-2000:] or r.stderr[-2000:])
    shutil.copy(sheet, os.path.join(MANUAL, "timing_handcheck_50.csv"))

    d = pd.read_csv(sheet)
    if "sources_matching_header" in d.columns:
        print()
        print("which source the filing header agrees with, on the fifty")
        for label in ("edgar", "ibes", "wire"):
            k = d.sources_matching_header.fillna("").str.contains(label).sum()
            print(f"  {label:<6} {k:>3} of {len(d)}")
        print()
        print(d.source_you_trust.value_counts().to_string())
        print()
        print("These are conflicts by construction, so the rates are not")
        print("the rates over the whole sample. What they say is which")
        print("source is late when the three disagree.")
        d.to_csv(os.path.join(RESULTS, "timing_handcheck_50_judged.csv"),
                 index=False)

50 events to settle
  10/50
  20/50
  30/50
  40/50
  50/50

source_you_trust
wire     44
edgar     6

settled from the header: 50/50
  edgar  matches the header in  48 of 50
  ibes   matches the header in  11 of 50
  wire   matches the header in   4 of 50

-> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\files csv\timing_handcheck_50.csv


which source the filing header agrees with, on the fifty
  edgar   48 of 50
  ibes    11 of 50
  wire     4 of 50

source_you_trust
wire     44
edgar     6

These are conflicts by construction, so the rates are not
the rates over the whole sample. What they say is which
source is late when the three disagree.


In [22]:
# 8. THE FIGURES
# Four charts into results/. The funnel and the timing disagreements are
# the two the report leans on; the other two say where the sample sits in
# time and how far the wire moves a session.

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fn = pd.read_csv(f"{DATA}/attrition_funnel.csv")
rc = pd.read_csv(f"{DATA}/raw_counts.csv")
ms = pd.read_csv(f"{DATA}/matched_sample.csv",
                 usecols=["session_date"])
ts = pd.read_csv(f"{DATA}/three_source_timing.csv",
                 usecols=["edgar_session", "wire_session", "ibes_session_c"])

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

a = ax[0][0]
a.barh(fn.step[::-1], fn.remaining[::-1], color="#4a5568")
a.set_title("Attrition funnel")
a.set_xlabel("filings remaining")
for i, n in enumerate(fn.remaining[::-1]):
    a.text(n, i, f" {n:,}", va="center", fontsize=8)
a.tick_params(axis="y", labelsize=8)

a = ax[0][1]
y = rc[rc.year != "TOTAL"].copy()
y["year"] = y.year.astype(int)
a.plot(y.year, y.item_2_02_filings, marker="o", color="#2c5282",
       label="Item 2.02 filings")
ev_per_year = (pd.to_datetime(ms.session_date, errors="coerce")
                 .dt.year.value_counts().sort_index())
a.plot(ev_per_year.index, ev_per_year.values, marker="o", color="#9b2c2c",
       label="in the final sample")
a.set_title("Filings a year, before and after the filters")
a.legend(frameon=False)

a = ax[1][0]
# Each rate is over the events where both sources speak. Dividing by every
# event instead would mix "the two disagree" with "one is silent", which
# are different facts and move the number by ten points.
def agree(a, b):
    both = ts[a].notna() & ts[b].notna()
    return (ts.loc[both, a] == ts.loc[both, b]).mean(), int(both.sum())


pairs = {f"EDGAR = I/B/E/S\nn={agree('edgar_session', 'ibes_session_c')[1]:,}":
         agree("edgar_session", "ibes_session_c")[0],
         f"EDGAR = wire\nn={agree('edgar_session', 'wire_session')[1]:,}":
         agree("edgar_session", "wire_session")[0],
         f"I/B/E/S = wire\nn={agree('ibes_session_c', 'wire_session')[1]:,}":
         agree("ibes_session_c", "wire_session")[0]}
a.bar(list(pairs), [100 * v for v in pairs.values()], color="#4a5568")
a.set_ylim(0, 100)
a.set_ylabel("% of events in the same session")
a.set_title("Do two sources put the release in the same session?")
for i, v in enumerate(pairs.values()):
    a.text(i, 100 * v, f"{100 * v:.1f}%", ha="center", va="bottom",
           fontsize=9)

a = ax[1][1]
delta = (pd.to_datetime(ts.wire_session, errors="coerce")
         - pd.to_datetime(ts.edgar_session, errors="coerce")).dt.days
d = delta[(delta.notna()) & (delta != 0) & (delta.abs() <= 5)]
a.bar(d.value_counts().sort_index().index,
      d.value_counts().sort_index().values, color="#9b2c2c")
a.set_title("How far the wire moves the session")
a.set_xlabel("sessions, wire minus EDGAR")

for row in ax:
    for a in row:
        a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
path = os.path.join(RESULTS, "nov2_figures.png")
fig.savefig(path, dpi=150)
print("->", path)

-> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\results\1_results\nov2_figures.png


In [23]:
# 9. ACCESS, CONFIRMED BY QUERY RATHER THAN FROM THE CATALOGUE
# An entitlement that renders in the web interface can still refuse a query,
# and a schema that exists can still be empty for us.

QUERIES = {
    "crsp.dsf_v2": "select max(dlycaldt) from crsp.dsf_v2",
    "crsp.dsf": "select max(date) from crsp.dsf",
    "ibes.det_guidance": "select count(*) from ibes.det_guidance "
                         "where anndats >= '2024-01-01'",
    "ibes.actu_epsus": "select max(anndats) from ibes.actu_epsus",
    "ibes.detu_epsus": "select max(anndats) from ibes.detu_epsus",
    "wrdsapps.ibcrsphist": "select count(*) from wrdsapps.ibcrsphist "
                           "where score <= 3",
    "crsp.ccmxpf_lnkhist": "select count(*) from crsp.ccmxpf_lnkhist",
    "wrdssec.wciklink_gvkey": "select count(*) from wrdssec.wciklink_gvkey",
}

if RUN_FROM_SCRATCH:
    %pip install -q wrds
    import wrds
    db = wrds.Connection(wrds_username=WRDS_USER)
    for table, q in QUERIES.items():
        try:
            print(f"{table:<26}{db.raw_sql(q).iloc[0, 0]}")
        except Exception as exc:
            print(f"{table:<26}refused: {type(exc).__name__}")
    db.close()
else:
    for table, q in QUERIES.items():
        print(f"{table:<26}{q}")

crsp.dsf_v2               select max(dlycaldt) from crsp.dsf_v2
crsp.dsf                  select max(date) from crsp.dsf
ibes.det_guidance         select count(*) from ibes.det_guidance where anndats >= '2024-01-01'
ibes.actu_epsus           select max(anndats) from ibes.actu_epsus
ibes.detu_epsus           select max(anndats) from ibes.detu_epsus
wrdsapps.ibcrsphist       select count(*) from wrdsapps.ibcrsphist where score <= 3
crsp.ccmxpf_lnkhist       select count(*) from crsp.ccmxpf_lnkhist
wrdssec.wciklink_gvkey    select count(*) from wrdssec.wciklink_gvkey
